# Reproducing van der Molen, Spaeth, Chini et al. (2025) from dandiset 001603

**Paper.** van der Molen T, Spaeth A, Chini M, et al. (2025). Preconfigured neuronal firing
sequences in human brain organoids. *Nature Neuroscience*.
[doi:10.1038/s41593-025-02111-0](https://doi.org/10.1038/s41593-025-02111-0). The claims file was
written from the NIH author manuscript in PubMed Central (PMC12912721.1) with the publisher's
Supplementary Information; this attempt read the same text.

**Dandiset.** [DANDI:001603](https://dandiarchive.org/dandiset/001603/0.260923.0623), version
0.260923.0623. The paper cites no version of the dandiset. This is the only published version and
the most recent one; the draft has the same 111 assets and size, so the published version is
pinned.

**What is used.** The sorted files only: each holds a 1 ms binary spike matrix
(`processing/curated_binning/t_spk_mat`, the input of the paper's code), and most also a units
table. Recordings are cut to the code's lengths (180 s for HO1 to HO4, slices and primary
cultures; 600 s for HO5 to HO8 and murine organoids). The raw `_ecephys` files are not read.

**Method sources.** The claims file (its opening comment and each criterion), and where it is
silent, the paper's Methods and Supplementary Methods and the code it links
(github.com/braingeneers/Protosequences at e9a79c4), cited in `sequences.py` and below.

**Shuffles.** The paper normalises to 100 Okun-swap shuffles per recording. This notebook uses 10
per recording, and 3 for the pairwise cross-correlations and the shuffled burst similarity, to fit
the runtime limit (a deviation; the shuffle mean is the reference, and its noise is small next to
the effects tested). Every shuffle made all the swaps requested (data check).

**Bursts.** Peaks at 700 ms minimum distance (Methods); where a peak in the tail of a long burst
gives a second, overlapping burst, only the burst with the higher peak is kept, which is what the
Methods' increase of the distance "up to 2000 ms" for long-burst recordings is for (an assumption
on which recordings it applies to; 2000 ms for every recording is shown as a variant).

**Statistics.** Mixed models are fitted by REML with a random intercept, and fixed effects and
contrasts are tested with t on Satterthwaite degrees of freedom, as lmerTest does
(`sequences.lmm`).

| Claim | Tested here |
|---|---|
| methods-burst-count, fig1c-example-burst-count, ed1c-lognormal-rates, fig1d-burst-duration, fig1e-peaks-near-burst-peak | tested |
| fig2-backbone-fraction, fig2e-b2b-correlation, fig2g-variance-latency, ed3f-burst-cluster-variability | tested |
| ed4b-gabazine-more-bursts, ed4b-nbqx-cpp-abolish-bursts, ed4c-gabazine-participation, ed4d-gabazine-rank-order | tested |
| supp2a-backbone-fraction-declines, supp2c-b2b-increases-with-age | tested on the human organoids only (murine longitudinal recordings are not in the dandiset) |
| supp8-backbone-stable-4h | tested, with units linked by their identical positions (an assumption) |
| ed5-shuffle-reduces-b2b, fig3e-pairwise-correlation, fig3f-backbone-pairs-in-tail | tested |
| fig4c-similarity-variance, fig4d-burst-similarity, fig4e-similarity-rises-at-backbone-onset, supp13-top-correlated-units-similarity, fig4f-pca-variance | tested |
| fig6f-slice-b2b-correlation, fig6g-slice-pairwise-correlation | tested |
| fig7a to fig7e (ten claims), supp24c-nonrigid-more-variable, supp10b-rank-order-by-model | tested, under both readings of the recording length |
| ed2d-alam-el-din-rates, ed2e-alam-el-din-b2b, ed2f-backbone-pairs-above-zero | tested |
| HMM claims (supp18, fig5c, fig5d, fig5f, fig7f, supp24d, supp26) | in `hmm_van_der_molen_2025.ipynb` |
| supp27-d2-near-critical | tested, with the d2 code of Sooter et al. ported |
| supp5-murine-organoid-forebrain-identity | blocked: data elsewhere (GEO), markers not listed |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from scipy import stats

import sequences as sq

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

assets = sq.list_assets()
sorted_assets = [a for a in assets if "_ecephys" not in a["path"]]
with ThreadPoolExecutor(8) as ex:
    infos = list(ex.map(sq.sorted_file_info, sorted_assets))
files = sq.recording_table(assets, infos)
print(f"dandiset {sq.DANDISET} version {sq.VERSION}: {len(assets)} assets, "
      f"{len(sorted_assets)} sorted files, {len(assets) - len(sorted_assets)} raw files")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print(files[["path", "recording", "subject", "age", "species", "frames", "n_units", "has_units_table"]]
      .to_string(index=False))
dups = files.groupby("recording").path.apply(list)
print("\nRecordings deposited more than once:")
print(dups[dups.str.len() > 1].to_string())


def matrix_hash(asset_id):
    import hashlib
    m = sq.load_matrix(asset_id)
    return hashlib.sha1(np.packbits(m).tobytes() + str(m.shape).encode()).hexdigest()[:12]


with ThreadPoolExecutor(8) as ex:
    files["hash"] = list(ex.map(matrix_hash, files.asset_id))
same = files.groupby("hash").recording.apply(lambda s: sorted(set(s)))
print("\nGroups of files with identical spike matrices (more than one file):")
for h, recs in same.items():
    n = (files.hash == h).sum()
    if n > 1:
        print(f"  {n} files: {recs}")

Each of the HO2 and HO3 recordings at 6, 7 and 8 months and 8 months plus 4 h is deposited twice,
and the 7-month recordings a third time as `HO2_sorted.mat` and `HO3_sorted.mat`; the hashes above
show which copies hold identical spike matrices. Each recording is counted once (the first file
by path). The murine organoid MO12 has gabazine and NBQX with R-CPP sorted files but no baseline
sorted file, and no murine organoid longitudinal recording is in the dandiset.

In [ ]:
first = files.sort_values("path").groupby("recording").first()


def check_units(rec):
    row = first.loc[rec]
    m = sq.load_matrix(row.asset_id)
    st = sq.load_units_table(row.asset_id)
    out = dict(recording=rec, units=m.shape[0], seconds=m.shape[1] / 1000, spikes_matrix=int(m.sum()))
    if st is not None:
        n_tab = np.array([len(x) for x in st])
        n_mat = m.sum(1)
        out.update(spikes_table=int(n_tab.sum()),
                   units_table_max_s=max((x.max() for x in st if len(x)), default=np.nan),
                   units_matrix_le_table=bool((n_mat <= n_tab).all()),
                   lost_to_1ms_bins=int((n_tab - n_mat).sum()))
    return out


with ThreadPoolExecutor(8) as ex:
    chk = pd.DataFrame(list(ex.map(check_units, first.index)))
print(chk.to_string(index=False))
print("\nEvery matrix count is at most the units-table count (several spikes in one 1 ms bin count once):",
      bool(chk.units_matrix_le_table.dropna().all()))

The spike matrices are 1 ms bins (the file's description says so); where a units table exists,
each unit's matrix count is at most its spike count in the table, the difference being spikes
that fall in the same millisecond, and the table's last spike lies within the matrix length.
Some units tables hold a single NaN for units with no spikes. The species field says *Rattus
norvegicus* for Pr1 to Pr4 (Yuan et al., E18 Wistar rat) and *Mus musculus* for Pr5 to Pr8; the
paper calls all primary cultures murine. All files share one clock per recording: there is no
trial table, and every measure here is computed from the spike matrix alone, so a check of
alignment between streams does not apply.

In [ ]:
RECS = sq.MAIN


def run(args):
    name, rec, n_shuf, n_pair, sim, seed, light = args
    m = sq.load_matrix(first.loc[rec].asset_id, sq.keep_seconds(name))
    t0 = time.time()
    r = sq.analyse(name, m, seed, n_shuf=n_shuf, n_shuf_pair=n_pair, similarity=sim, light=light)
    if rec == name:
        # d2 (supp27-d2-near-critical): intact, one per-unit shuffle, and the AR fit check
        rng = np.random.default_rng(seed + 500)
        x = sq.bin_pop(m)
        vp, vn = sq.ar_yule_walker(x)
        xs = sq.bin_pop(sq.shuffle_each_unit(m, rng))
        r["d2"] = dict(d2=sq.d2_distance(vp), d2_shuffled=sq.d2_distance(sq.ar_yule_walker(xs)[0]),
                       **sq.ar_fit_check(x, vp, vn, rng))
    r["secs"] = time.time() - t0
    return rec, r


jobs = [(n, n, 10, 3, n.startswith("HO"), 1000 + i, False) for i, n in enumerate(RECS)]
jobs += [(n, f"{n}_{c}", 10, 0, False, 2000 + 10 * i + j, True)
         for i, (n, cs) in enumerate(sq.DRUG.items()) for j, c in enumerate(cs) if f"{n}_{c}" in first.index]
jobs += [(n, f"{n}_{c}", 0, 0, False, 3000, False) for n, cs in sq.LONGITUDINAL.items() for c in cs]
res = {}
with ProcessPoolExecutor(2) as ex:
    for rec, r in ex.map(run, jobs):
        res[rec] = r
R = {k: res[k] for k in RECS}
D = {k: v for k, v in res.items() if k.split("_")[0] in sq.DRUG and "_" in k}
L = {k: v for k, v in res.items() if k.split("_")[0] in sq.LONGITUDINAL and "_" in k}
print(f"analysed {len(res)} recordings in {time.time() - started:.0f} s")

In [ ]:
summ = pd.DataFrame([dict(recording=k, type=r["mtype"], units=r["U"], seconds=r["T"] / 1000,
                          units_ge30_spikes=int((r["n_spk"] >= 30).sum()),
                          mean_rate_hz=round(r["rate"].mean(), 3), max_rate_hz=round(r["rate"].max(), 2),
                          bursts=r["n_bursts"], overlapping_dropped=r["n_overlap"],
                          backbone=int(r["bb"].sum()) if "bb" in r else np.nan,
                          shuffles=len(r.get("shuf", {}).get("done", [])))
                     for k, r in res.items()])
print(summ.to_string(index=False))
print("\nRecordings per model type (main set):", pd.Series([R[k]["mtype"] for k in RECS]).value_counts().to_dict())
print("Paper: 8 human organoids, 9 murine organoids, 6 slices, 8 primary cultures.")
print("Burst parameters:", sq.BURST_PARAMS, "minimum peak distance", sq.MIN_PEAK_DIST_MS, "ms")
print("Overlapping bursts dropped (a peak in the tail of a longer burst):", int(summ.overlapping_dropped.sum()),
      "in", int((summ.overlapping_dropped > 0).sum()), "recordings")

In [ ]:
# Burst threshold against the coarse population rate (the rate burst peaks are found on)
diag = pd.DataFrame([dict(recording=k, bursts=R[k]["n_bursts"], rms=R[k]["rms"], threshold=R[k]["thr"],
                          median_rate=R[k]["coarse_median"], highest_peak=R[k]["coarse_top"][0],
                          highest_peak_over_rms=R[k]["coarse_top"][0] / R[k]["rms"],
                          top5_peaks=np.round(R[k]["coarse_top"], 2).tolist()) for k in RECS])
print(diag.round(3).to_string(index=False))
print("\nPopulation rate in spikes per 1 ms frame summed over units (20 ms moving mean, 100 ms Gaussian; 50 ms for Pr).")

# Okun swaps made against those requested (5 per spike); a swap is refused when the two units or
# frames are equal or the target bins already hold a spike, and the loop stops after 4 x (requested
# + spikes) draws
sw = pd.DataFrame([dict(recording=k, shuffles=len(r["shuf"]["done"]), requested=r["shuf"]["requested"],
                        done_min=min(r["shuf"]["done"]), done_frac_min=min(r["shuf"]["done"]) / r["shuf"]["requested"])
                   for k, r in res.items() if r.get("shuf", {}).get("done")])
print(sw.round(3).to_string(index=False))
print("\nShuffles with fewer swaps than requested:", int((sw.done_frac_min < 1).sum()), "recordings; lowest fraction",
      round(sw.done_frac_min.min(), 3))

The dandiset holds 8 human organoids, 9 murine organoids, 6 slices and 8 primary cultures, as the
paper says. 35 overlapping bursts in 13 recordings (HO5 to HO8, MO9, the slices, Pr7) are peaks in
the tail of a longer burst and are dropped (cell above). Following the stated burst detection
(population rate above 4 times its RMS), four murine organoids (MO2, MO4, MO5, MO6) have no burst
and MO3 has four. The table of thresholds shows why: their highest peaks reach only 3.0 to 3.7
times the RMS of the rate, against 4.2 to 18 times in every other recording, because their rate
between bursts is high (median coarse rate 0.47 to 1.07 spikes per ms, against highest peaks of
4.1 to 6.8), so the RMS is close to the peaks. The paper's code processed every murine organoid, so it must have used
other burst parameters for them, which the paper does not give. Claims that rest on murine
organoid bursts use the recordings that have bursts, and say so. Every shuffle made all the Okun
swaps requested (5 per spike).

## methods-burst-count

> The analysed segment of each recording holds about 40 population bursts (38 ± 4, mean ± SE).

In [ ]:
nb = pd.Series({k: R[k]["n_bursts"] for k in RECS})
nb800 = pd.Series({k: R[k]["n_bursts_800"] for k in RECS})
nb2000 = pd.Series({k: R[k]["n_bursts_2000"] for k in RECS})
print(pd.DataFrame({"bursts_700ms": nb, "bursts_800ms": nb800, "bursts_2000ms": nb2000,
                    "overlapping_dropped": pd.Series({k: R[k]["n_overlap"] for k in RECS}),
                    "seconds": [R[k]["T"] / 1000 for k in RECS]}).T.to_string())
for lab, s in [("700 ms, overlapping bursts dropped (main)", nb), ("800 ms", nb800), ("2000 ms for every recording", nb2000)]:
    print(f"{lab}: mean {s.mean():.1f}, SE {s.std(ddof=1) / np.sqrt(len(s)):.1f}, SD {s.std(ddof=1):.1f}, range {s.min()} to {s.max()}")
ok = nb > 0
print(f"Leaving out the recordings with no burst: mean {nb[ok].mean():.1f}, SE {nb[ok].std(ddof=1) / np.sqrt(ok.sum()):.1f}")
print("Expected 38 +- 4")

**Result: reproduced, numerically.** The mean is 34.8 bursts over the 31 recordings (within 38 ±
4), with an SE of 6.3 and a range from 0 to 156: four murine organoids have no burst and two have
more than 100, so the mean matches while the recordings do not each hold about 40 bursts. The
agreement rests on MO1 and MO7 offsetting the four recordings without bursts. With 800 ms the mean
is 34.7; with 2000 ms for every recording it is 29.5 (outside), but the Methods raise the distance
only for recordings with long bursts.

## fig1c-example-burst-count

> In the example human organoid of Figs. 1 and 2, 46 bursts occur in a three-minute interval.

In [ ]:
print(pd.DataFrame({k: dict(min_dist_700=R[k]["n_bursts"], min_dist_800=R[k]["n_bursts_800"])
                    for k in ["HO1", "HO2", "HO3", "HO4"]}).to_string())
print("Expected 46 +- 4.6")

**Result: reproduced.** HO1 has 46 bursts in its first 180 s under both peak distances; HO1 is
the example organoid for the claims that depend on this one.

In [ ]:
EXAMPLE = "HO1"

## ed1c-lognormal-rates

> In each of the 8 human organoids the distribution of unit mean firing rates is well fit by a
> lognormal distribution (R² 0.97 ± 0.04), while the rates of backbone units alone are not (R²
> 0.45 ± 0.30).

In [ ]:
HO8 = [f"HO{i}" for i in range(1, 9)]
rows = []
for units in ["all", "ge30"]:
    for reading in ["a_lognormal", "a_code_variant", "b_legend"]:
        fn = {"a_lognormal": lambda x: sq.lognormal_r2(x),
              "a_code_variant": lambda x: sq.lognormal_r2(x, variant=True),
              "b_legend": sq.lognormal_r2_legend}[reading]
        for k in HO8:
            r = R[k]
            m = np.ones(r["U"], bool) if units == "all" else r["n_spk"] >= sq.MIN_SPIKES
            rate, bb = r["rate"][m], r["bb"][m]
            rows.append(dict(units=units, reading=reading, rec=k, n=len(rate), n_bb=int(bb.sum()),
                             r2_all=fn(rate), r2_bb=fn(rate[bb]) if bb.sum() >= 3 else np.nan))
e1 = pd.DataFrame(rows)
print(e1.pivot_table(index=["units", "reading"], columns="rec", values="r2_all").round(3).to_string())
print(e1.pivot_table(index=["units", "reading"], columns="rec", values="r2_bb").round(3).to_string())
print("\nMean (SD) over organoids; backbone fits need at least 3 backbone units (HO7 has none):")
print(e1.groupby(["units", "reading"]).agg(all_mean=("r2_all", "mean"), all_sd=("r2_all", "std"),
                                           bb_mean=("r2_bb", "mean"), bb_sd=("r2_bb", "std"),
                                           n_bb_fits=("r2_bb", "count")).round(3).to_string())
print("Expected: all 0.97 +- 0.04; backbone 0.45 +- 0.30")

**Result: reproduced, under reading (a) with all units.** The mean R² is 0.949 for all units and
0.487 for backbone units (seven organoids), both within tolerance; the code's form of the density
gives 0.944 and 0.442. With units of at least 30 spikes, all units give 0.919 (outside 0.97 ±
0.04). Reading (b), the legend's normal fit to log rates, gives means of 0.60 and 0.36.

## fig1d-burst-duration

> Burst durations in the human organoids are on the order of 10² ms.

In [ ]:
dur = pd.DataFrame({k: dict(median_ms=np.median(R[k]["duration"]), min_ms=R[k]["duration"].min(),
                            max_ms=R[k]["duration"].max(), bursts=R[k]["n_bursts"])
                    for k in RECS if R[k]["n_bursts"] >= 2}).T
print(dur.to_string())
in_decade = dur.loc[["HO1", "HO2", "HO3", "HO4"], "median_ms"].between(100, 1000)
print("\nHO1 to HO4 median in 100 to 1000 ms:", in_decade.to_dict())

**Result: reproduced.** The median burst durations of HO1 to HO4 are 511, 628, 472 and 892 ms,
all within 100 to 1000 ms. Over all recordings with bursts the medians run from 104 ms (Pr4) to
2681 ms (MO3); HO6, MO3 and MO9 have medians above 1000 ms.

## fig1e-peaks-near-burst-peak

> Most units reach their peak firing rate close to the peak of the population burst.

In [ ]:
rows = []
for k in ["HO1", "HO2", "HO3", "HO4"]:
    P = R[k]["P"]
    med = np.nanmedian(P, 1)
    med = med[np.isfinite(med)]
    pooled = P[np.isfinite(P)]
    for bound in [50, 100]:
        rows.append(dict(rec=k, bound_ms=bound, units=len(med), median_within=np.mean(np.abs(med) <= bound),
                         unit_bursts=len(pooled), pooled_within=np.mean(np.abs(pooled) <= bound)))
f1e = pd.DataFrame(rows)
print(f1e.round(3).to_string(index=False))
print("\nReading and organoid combinations above one half:",
      int((f1e.median_within > 0.5).sum() + (f1e.pooled_within > 0.5).sum()), "of", 2 * len(f1e))

**Result: partially reproduced.** In 15 of the 16 combinations of organoid, quantity and bound,
more than half of the values lie within the bound (0.53 to 0.89); the exception is HO4 with
pooled unit-burst peak times within 50 ms (0.481). The criterion requires every reading, so the
claim holds in part.

## fig2-backbone-fraction

> Backbone units, which fire at least twice in every burst, are 28% ± 14% (mean ± STD) of the
> units in the 8 human organoids.

In [ ]:
bf = pd.DataFrame({k: dict(units=R[k]["U"], units_ge30=int((R[k]["n_spk"] >= 30).sum()),
                           backbone=int(R[k]["bb"].sum()), frac_all=R[k]["bb"].mean(),
                           frac_ge30=R[k]["bb"][R[k]["n_spk"] >= 30].mean()) for k in HO8}).T
print(bf.to_string())
for c in ["frac_all", "frac_ge30"]:
    print(f"{c}: mean {bf[c].mean():.3f}, SD {bf[c].astype(float).std(ddof=1):.3f} (population SD {bf[c].astype(float).std(ddof=0):.3f})")
print("Expected mean 0.28 +- 0.14, SD 0.14 +- 0.014")

**Result: reproduced, under reading (b).** With units of at least 30 spikes as the denominator
the mean is 0.331 and the SD 0.150, both within tolerance. With all sorted units the mean is
0.258 (within) and the SD 0.166 (outside 0.126 to 0.154). HO7 has no backbone unit here (no unit fires twice in all 21 bursts).

## fig2e-b2b-correlation

> Backbone units have higher burst-to-burst correlations of their firing rate than non-rigid units.

In [ ]:
def unit_table(keys, bbkey="bb", src=None):
    src = R if src is None else src
    rows = []
    for k in keys:
        r = src[k]
        if "bb" not in r:
            continue
        sh = r.get("shuf", {})
        b2b_sh = np.nanmean(np.array(sh["b2b30"]), 0) if sh.get("b2b30") else np.full(r["U"], np.nan)
        cv_sh = np.array(sh["cv2"]) if sh.get("cv2") else None
        for u in range(r["U"]):
            rows.append(dict(rec=k, mtype=r["mtype"], unit=f"{k}_{u}", n_spk=r["n_spk"][u], rate=r["rate"][u],
                             bb=int(r[bbkey][u]), b2b30=r["b2b30"][u], b2b70=r["b2b70"][u], b2b_shuf=b2b_sh[u],
                             cv2=r["cv2"][u],
                             cv2_shuf_mean=np.nanmean(cv_sh[:, u]) if cv_sh is not None else np.nan,
                             cv2_shuf_sd=np.nanstd(cv_sh[:, u], ddof=1) if cv_sh is not None else np.nan))
    return pd.DataFrame(rows)


HO4 = ["HO1", "HO2", "HO3", "HO4"]
ut = unit_table(HO4)
for rule in ["b2b30", "b2b70"]:
    d = ut[np.isfinite(ut[rule])].rename(columns={rule: "y"})
    f = sq.lmm(d, "y ~ bb", "rec")
    print(f"{rule}: {len(d)} units ({d.bb.sum()} backbone); mean backbone {d[d.bb == 1].y.mean():.3f}, "
          f"non-rigid {d[d.bb == 0].y.mean():.3f}; LMM effect {f.params['bb']:.3f}, t {f.tvalues['bb']:.1f}, p {f.pvalues['bb']:.2g}")
    print("   per organoid:", d.groupby(["rec", "bb"]).y.mean().round(3).unstack().to_dict("index"))
print(f"\nAll sorted units in HO1 to HO4: {sum(R[k]['U'] for k in HO4)} (expected 507 +- 51)")

**Result: reproduced.** Backbone units have higher burst-to-burst correlations than non-rigid
units (0.796 against 0.574, p = 3e-27, 216 units under the Methods' 30% rule; 0.796 against
0.629, p = 9e-16, 150 units under the code's 70% rule), in each organoid. The paper's n of 507
is the number of all sorted units in HO1 to HO4 (507 here as well); the units that pass the
rules are fewer, and burst-to-burst correlations exist only for units active in bursts, so a test
over all 507 cannot be run.

## fig2g-variance-latency

> Among backbone units, the variance of firing-rate peak times across bursts grows with the
> unit's position in the sequence.

In [ ]:
for reading in ["a_rank", "b_rescaled"]:
    rows = []
    for k in HO4:
        r = R[k]
        P = r["P"][r["bb"]]
        med = np.nanmedian(P, 1)
        var = np.nanvar(P, 1, ddof=1)
        if reading == "a_rank":
            pos = (stats.rankdata(med) - 1) / (len(med) - 1)
        else:
            pos = (med - med.min()) / (med.max() - med.min())
        rows += [dict(rec=k, pos=a, var=v) for a, v in zip(pos, var)]
    d = pd.DataFrame(rows)
    f = sq.lmm(d, "var ~ pos", "rec")
    fl = sq.lmm(d.assign(lv=np.log10(d["var"] + 1)), "lv ~ pos", "rec")
    print(f"{reading}: {len(d)} backbone units; slope {f.params['pos']:.0f} ms^2 per unit position, p {f.pvalues['pos']:.2g}; "
          f"log10 variance slope {fl.params['pos']:.2f}, p {fl.pvalues['pos']:.2g}")
    print("   Spearman per organoid:", {k: round(stats.spearmanr(g.pos, g["var"])[0], 2) for k, g in d.groupby("rec")})

**Result: reproduced.** The variance of backbone peak times rises with relative position under
both readings (rank, p = 6e-4; rescaled median time, p = 1e-4; on log variance p = 3e-6 and
1e-6), with positive rank correlations in each organoid (0.11 to 0.68).

## ed3f-burst-cluster-variability

> When bursts are clustered by their pairwise unit correlation matrices, the firing rates of
> non-rigid units vary more between clusters than those of backbone units.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def burst_corr_vectors(seg):
    """Per burst, the lower triangle of the unit-by-unit correlation of the -250 to 500 ms rates
    (numpy corrcoef; undefined correlations set to 0, as the code's nanmax with the identity)."""
    U = seg.shape[0]
    tri = np.tril_indices(U, -1)
    out = []
    for b in range(seg.shape[1]):
        with np.errstate(invalid="ignore", divide="ignore"):
            c = np.corrcoef(seg[:, b, :])
        c = np.nan_to_num(c, nan=0.0)
        out.append(c[tri])
    return np.array(out)


def kneedle(ks, sse):
    """Knee of a decreasing convex curve: the point farthest below the chord (Satopaa et al.)."""
    x = (np.array(ks) - ks[0]) / (ks[-1] - ks[0])
    y = (np.array(sse) - min(sse)) / (max(sse) - min(sse))
    return ks[int(np.argmax((1 - x) - y))]


rows, kinfo = [], []
for k in HO8:
    r = R[k]
    seg = r["segments"]
    V = burst_corr_vectors(seg)
    ks = list(range(2, min(10, len(V) - 1) + 1))
    sse, sil, labels = [], [], {}
    for kk in ks:
        best = None
        for seed in range(10):
            km = KMeans(n_clusters=kk, init="k-means++", random_state=seed, n_init=1).fit(V)
            if best is None or km.inertia_ < best.inertia_:
                best = km
        sse.append(best.inertia_)
        sil.append(silhouette_score(V, best.labels_))
        labels[kk] = best.labels_
    choice = {"a_kneedle": kneedle(ks, sse), "b_silhouette": ks[int(np.argmax(sil))]}
    kinfo.append(dict(rec=k, bursts=len(V), **choice))
    mean_rate = seg.mean(2)  # units by bursts
    for reading, kk in choice.items():
        lab = labels[kk]
        cm = np.array([mean_rate[:, lab == c].mean(1) for c in range(kk)]).T
        with np.errstate(invalid="ignore", divide="ignore"):
            cv = cm.std(1) / cm.mean(1)
        rows += [dict(reading=reading, rec=k, bb=int(r["bb"][u]), cv=cv[u]) for u in range(r["U"]) if np.isfinite(cv[u])]
print(pd.DataFrame(kinfo).to_string(index=False))
cvd = pd.DataFrame(rows)
for reading in ["a_kneedle", "b_silhouette"]:
    for recs, lab in [(HO8, "HO1-HO8"), (HO4, "HO1-HO4")]:
        d = cvd[(cvd.reading == reading) & cvd.rec.isin(recs)]
        f = sq.lmm(d, "cv ~ bb", "rec")
        print(f"{reading} {lab}: {len(d)} units; CV backbone {d[d.bb == 1].cv.mean():.3f}, non-rigid {d[d.bb == 0].cv.mean():.3f}; "
              f"effect of backbone {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: reproduced.** Non-rigid units vary more between burst clusters than backbone units
under both elbow readings and both organoid sets (mean CV 0.64 to 1.31 against 0.07 to 0.30, p
from 6e-40 to 2e-84). The elbow picked by the kneedle rule (3 to 7 clusters) and by the largest
silhouette (2 or 3) differ, and the outcome is the same.

## Murine organoids with gabazine and with NBQX and R-CPP (ed4b, ed4c, ed4d)

In [ ]:
drug = pd.DataFrame([dict(rec=k, organoid=k.split("_")[0], cond=k.split("_")[1], units=r["U"],
                          seconds=r["T"] / 1000, bursts=r["n_bursts"],
                          bursts_per_min=r["n_bursts"] / (r["T"] / 60000)) for k, r in D.items()])
print(drug.to_string(index=False))
print("\nMO12 has no baseline sorted file. Unit counts are equal across the conditions of each organoid.")

## ed4b-gabazine-more-bursts

> In murine organoids, blocking GABA-A receptors with gabazine increases the number of population
> bursts.

In [ ]:
d = drug[drug.cond.isin(["BASE", "GABAZINE"])].copy()
d["gab"] = (d.cond == "GABAZINE").astype(int)
print(d.pivot(index="organoid", columns="cond", values="bursts_per_min").round(2).to_string())
f = sq.lmm(d, "bursts_per_min ~ gab", "organoid")
print(f"\nBurst rate, gabazine minus baseline: {f.params['gab']:.2f} per min, p {f.pvalues['gab']:.2g} "
      f"({d.organoid.nunique()} organoids, {len(d)} recordings)")
pair = d.pivot(index="organoid", columns="cond", values="bursts_per_min").dropna()
print("Paired t-test over the organoids with both:", stats.ttest_rel(pair.GABAZINE, pair.BASE))
fc = sq.lmm(d, "bursts ~ gab", "organoid")
print(f"Burst count: {fc.params['gab']:.1f}, p {fc.pvalues['gab']:.2g}")

**Result: reproduced.** Gabazine raises the burst rate in each of the four organoids with a
baseline recording (by 2.7 to 10.6 bursts per minute; 7.1 per minute in the mixed model, p =
0.011; paired t-test over the four, p = 0.021).

## ed4b-nbqx-cpp-abolish-bursts

> Blocking AMPA and NMDA receptors with NBQX and R-CPP abolishes population bursts in murine
> organoids.

In [ ]:
d = drug[drug.cond.isin(["BASE", "NBQXCPP"])].copy()
d["nbqx"] = (d.cond == "NBQXCPP").astype(int)
print(d.pivot(index="organoid", columns="cond", values="bursts").to_string())
f = sq.lmm(d, "bursts_per_min ~ nbqx", "organoid")
print(f"\nBurst rate, NBQX with R-CPP minus baseline: {f.params['nbqx']:.2f} per min, p {f.pvalues['nbqx']:.2g}")
print("Treated recordings with at most one burst:", int((drug[drug.cond == "NBQXCPP"].bursts <= 1).sum()),
      "of", int((drug.cond == "NBQXCPP").sum()))

**Result: reproduced.** Every NBQX with R-CPP recording has no burst or one (MO12), against 81 to
114 bursts at baseline, and the fall is significant (10.1 per minute, p = 9e-7).

## ed4c-gabazine-participation

> Gabazine increases the fraction of bursts in which units fire at least two spikes.

In [ ]:
rows = []
for k, r in D.items():
    if "frac2" not in r or k.endswith("NBQXCPP"):
        continue
    o, c = k.split("_")
    rows += [dict(organoid=o, cond=c, unit=f"{o}_{u}", frac=r["frac2"][u], gab=int(c == "GABAZINE"))
             for u in range(r["U"])]
pd4 = pd.DataFrame(rows)
print(pd4.groupby(["organoid", "cond"]).frac.agg(["count", "mean"]).round(3).unstack().to_string())
f = sq.lmm(pd4, "frac ~ gab", "organoid")
print(f"\nPooled per condition (main reading): gabazine minus baseline {f.params['gab']:.3f}, p {f.pvalues['gab']:.2g}")
m = pd4[pd4.organoid.isin(["MO10", "MO11", "MO14"])]
fm = sq.lmm(m, "frac ~ gab", "organoid")
fu = sq.lmm(m, "frac ~ gab", "unit")
print(f"Variant, MO10, MO11 and MO14 (unit positions identical across conditions), random intercept per unit: "
      f"{fu.params['gab']:.3f}, p {fu.pvalues['gab']:.2g}")

# Unit positions across the conditions of each organoid (the dataset has no unit identifier)
for o in sq.DRUG:
    ks = [f"{o}_{c}" for c in sq.DRUG[o] if f"{o}_{c}" in first.index]
    pos = {k: sq.unit_positions(first.loc[k].asset_id) for k in ks}
    same = [f"{a.split('_')[1]} = {b.split('_')[1]}: {pos[a].shape == pos[b].shape and bool((pos[a] == pos[b]).all())}"
            for i, a in enumerate(ks) for b in ks[i + 1:]]
    print(f"{o}: units {[len(pos[k]) for k in ks]} ({', '.join(c.split('_')[1] for c in ks)}); identical ordered positions, {'; '.join(same)}")

**Result: reproduced.** With units pooled per condition, the fraction of bursts in which a unit
fires at least twice rises with gabazine by 0.139 (p = 9e-4), in each of the four organoids with
a baseline. Treating units as matched in MO10, MO11 and MO14, whose ordered unit positions are identical
between baseline and gabazine (last lines above; the dataset does not state that units are
matched), gives 0.152, p = 7e-8; MO12 has no baseline file and MO13's positions differ between
conditions, so both are left out of that variant.

## ed4d-gabazine-rank-order

> Gabazine makes the order of backbone sequences more consistent across bursts.

In [ ]:
def rank_table(src, keys):
    rows = []
    for k in keys:
        r = src[k]
        if r.get("rank") is None or not r["shuf"]["rank"]:
            continue
        A = r["rank"]
        S = np.array(r["shuf"]["rank"])
        B = np.nanmean(S, 0)
        sd = np.nanstd(S, 0, ddof=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            z = (A - B) / sd
        rows.append(pd.DataFrame(dict(rec=k, raw=A, shuf=B, norm=sq.normalize(A, B), z=z)))
    return pd.concat(rows)


rk = rank_table(D, [k for k in D if not k.endswith("NBQXCPP")])
rk["organoid"] = rk.rec.str.split("_").str[0]
rk["gab"] = rk.rec.str.endswith("GABAZINE").astype(int)
print(rk.groupby("rec")[["raw", "shuf", "norm", "z"]].median().round(3).to_string())
for v in ["norm", "z"]:
    d = rk[np.isfinite(rk[v])]
    f = sq.lmm(d, f"{v} ~ gab", "organoid")
    print(f"{v}: {len(d)} burst pairs; gabazine minus baseline {f.params['gab']:.3f}, p {f.pvalues['gab']:.2g}")
print("\nThe ratio (A - B)/(A + B) on signed correlations:")
print(f"   shuffle mean B: median {rk.shuf.median():.3f}, 5th to 95th percentile {rk.shuf.quantile(0.05):.3f} to {rk.shuf.quantile(0.95):.3f}")
print(f"   burst pairs with |A + B| < 0.1: {np.mean(np.abs(rk.raw + rk.shuf) < 0.1):.3f}; with A + B < 0 (sign reversed): {np.mean(rk.raw + rk.shuf < 0):.3f}")
print(f"   ratio 1st to 99th percentile {rk.norm.quantile(0.01):.1f} to {rk.norm.quantile(0.99):.1f}; min {rk.norm.min():.3g}, max {rk.norm.max():.3g}")
f_raw = sq.lmm(rk, "raw ~ gab", "organoid")
print(f"Raw correlation, gabazine minus baseline (no normalisation): {f_raw.params['gab']:.3f}, p {f_raw.pvalues['gab']:.2g}")

**Result: recorded as reproduced, on the assumption that the z-score normalisation is the one
used; a change to blocked (methods_underspecified) is proposed for the third verification round.**
The raw rank-order correlation of backbone sequences rises with gabazine in every organoid (medians
0.07 to 0.35 at baseline, 0.55 to 0.91 with gabazine; 0.381, p printed as 0), and normalised by
z-score to the shuffles gabazine is higher by 2.0 (p printed as 0). Normalised as (A - B)/(A + B)
it is not (-1.15, p = 0.6). Per burst pair that ratio is unstable: the shuffled Spearman
correlations are signed and centred on 0 (median 0.009, 5th to 95th percentile -0.12 to 0.13), so
A + B changes sign for 8.8% of burst pairs and is below 0.1 in magnitude for 5.8%, and the ratio
runs from -86,000 to 3,200. Its per-recording medians are stable, though (0.855 to 0.996), and
rise with gabazine in MO10, MO11 and MO13 and fall slightly in MO14 (0.959 to 0.953), so the ratio
reading has not been shown to carry no information. The claims file reads the paper correctly: the
Supplementary Methods name (A - B)/(A + B) for this measure and the Extended Data Fig. 4D legend
says z-scored, and the criterion requires gabazine to be higher under both readings. The readings
disagree, so the claim is underspecified; the z-score adopted in round 1 overrides the criterion.
Which normalisation the authors used is asked of them (a methods_unavailable finding).

## supp2a-backbone-fraction-declines and supp2c-b2b-increases-with-age (human organoids)

In [ ]:
lg = pd.DataFrame([dict(rec=k, organoid=k.split("_")[0], age=k.split("_", 1)[1], units=r["U"], bursts=r["n_bursts"],
                        backbone=int(r["bb"].sum()), frac_all=r["bb"].mean(),
                        frac_ge30=r["bb"][r["n_spk"] >= 30].mean()) for k, r in L.items()])
print(lg.to_string(index=False))
dev = lg[lg.age.isin(["6M", "7M", "8M"])]
for c in ["frac_all", "frac_ge30"]:
    t = dev.pivot(index="organoid", columns="age", values=c)
    print(f"\n{c}:\n{t.round(3).to_string()}")
    print("  (a) mean lower at 8M than 6M:", bool(t["8M"].mean() < t["6M"].mean()),
          "; (b) lower in every organoid:", bool((t["8M"] < t["6M"]).all()))
print("\nMurine organoid longitudinal recordings: none in the dandiset (MO1 to MO14 are all age P45D).")

**Result: partially reproduced.** In the human organoids HO2 and HO3 the backbone fraction is
lower at 8 months than at 6 months, on average and in each organoid, with either denominator.
The murine part cannot be tested: the dandiset holds no longitudinal murine organoid recordings.

In [ ]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

rows = []
for k, r in L.items():
    o, age = k.split("_", 1)
    if age not in ("6M", "7M", "8M"):
        continue
    rows += [dict(organoid=o, age=age, months=int(age[0]), bb=int(r["bb"][u]), y=r["b2b30"][u])
             for u in range(r["U"]) if np.isfinite(r["b2b30"][u])]
ag = pd.DataFrame(rows)
print(ag.groupby(["organoid", "age", "bb"]).y.agg(["count", "mean"]).round(3).unstack().to_string())
print(ag.groupby(["age", "bb"]).y.mean().round(3).unstack().to_string())
for second in ["C(bb)", "C(organoid)"]:
    fit = ols(f"y ~ C(age) + {second}", ag).fit()
    print(f"\nTwo-way ANOVA, age and {second}:\n", sm.stats.anova_lm(fit, typ=2).to_string(float_format=lambda v: f"{v:.3g}"))
mono = ag.groupby("age").y.mean()
print("\nMean by age:", mono.round(3).to_dict())
print("Backbone above non-rigid at every age:", bool((ag.groupby(["age", "bb"]).y.mean().unstack().diff(axis=1)[1] > 0).all()))

**Result: partially reproduced.** In HO2 and HO3 the burst-to-burst correlation rises with age
(means 0.61, 0.69 and 0.72 at 6, 7 and 8 months; age effect F = 18.7, p = 2e-8 with unit type as
the second factor, F = 13.3, p = 3e-6 with organoid), and backbone units are above non-rigid
units at every age. The murine part cannot be tested.

## supp8-backbone-stable-4h

> Units classed as backbone in one recording are mostly classed as backbone again 4 h later, units
> classed as non-rigid are mostly classed as non-rigid again, and backbone units keep more
> consistent firing patterns across the two recordings than non-rigid units.

The dandiset gives no unit identifier across recordings. The units are linked here by position:
walking both units tables in order, each 0 h unit is linked to the next 4 h unit at the identical
(x, y) position. Positions alone are not unique (several units share an electrode), but the whole
ordered list of positions is identical at 0 h and 4 h (HO3's 4 h file has one extra unit), which
is what joint sorting of the two recordings gives and what two separate sortings would not. This is
an assumption, and each recording's own result is shown as well. The cross-recording
burst-to-burst correlation pairs every 0 h burst with every 4 h burst (`cross_burst_to_burst`).

In [ ]:
x8 = []
for o in ["HO2", "HO3"]:
    k0, k4 = f"{o}_8M", f"{o}_8M_4H"
    p0, p4 = sq.unit_positions(first.loc[k0].asset_id), sq.unit_positions(first.loc[k4].asset_id)
    # walk both lists in order: a 0 h unit is linked to the next 4 h unit at the identical position,
    # skipping 4 h units that have no 0 h counterpart
    link, j = np.full(len(p0), -1), 0
    for u in range(len(p0)):
        while j < len(p4) and not (p4[j] == p0[u]).all():
            j += 1
        if j < len(p4):
            link[u], j = j, j + 1
    unlinked4 = sorted(set(range(len(p4))) - set(link[link >= 0]))
    n_pos = len({tuple(p) for p in p0})
    print(f"{o}: {len(p0)} units at 0 h ({n_pos} distinct positions), {len(p4)} at 4 h; 0 h units linked in order: "
          f"{int((link >= 0).sum())}; 4 h units left out: {unlinked4}")
    r0, r4 = L[k0], L[k4]
    ok = link >= 0
    bb0, bb4 = r0["bb"][ok], r4["bb"][link[ok]]
    print(f"   bursts {r0['n_bursts']} (0 h) and {r4['n_bursts']} (4 h); backbone units {int(r0['bb'].sum())} and {int(r4['bb'].sum())}")
    print(pd.crosstab(pd.Series(np.where(bb0, "backbone", "non-rigid"), name="0 h"),
                      pd.Series(np.where(bb4, "backbone", "non-rigid"), name="4 h")).to_string())
    stay_bb = bb4[bb0].mean() if bb0.any() else np.nan
    stay_nr = (~bb4[~bb0]).mean()
    M0 = sq.load_matrix(first.loc[k0].asset_id, sq.keep_seconds(o))
    M4 = sq.load_matrix(first.loc[k4].asset_id, sq.keep_seconds(o))[link[ok]]
    M0 = M0[ok]
    for rule in [0.3, 0.7]:
        c = sq.cross_burst_to_burst(M0, r0["bursts"], M4, r4["bursts"], rule)
        f = np.isfinite(c)
        d = pd.DataFrame(dict(y=c[f], bb=bb0[f].astype(int)))
        t = stats.ttest_ind(d.y[d.bb == 1], d.y[d.bb == 0], equal_var=False)
        x8.append(dict(organoid=o, rule=rule, backbone_stay=stay_bb, nonrigid_stay=stay_nr, units=int(f.sum()),
                       backbone_units=int(d.bb.sum()), cross_b2b_backbone=d.y[d.bb == 1].mean(),
                       cross_b2b_nonrigid=d.y[d.bb == 0].mean(), welch_p=t.pvalue))
    # each recording on its own
    for k, r in [(k0, r0), (k4, r4)]:
        y = r["b2b30"]
        print(f"   {k} alone, burst-to-burst (30% rule): backbone {np.nanmean(y[r['bb']]):.3f}, non-rigid {np.nanmean(y[~r['bb']]):.3f}")
x8 = pd.DataFrame(x8)
print(x8.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriterion: backbone_stay > 0.5, nonrigid_stay > 0.5 and cross_b2b_backbone > cross_b2b_nonrigid in both organoids:",
      bool(((x8.backbone_stay > 0.5) & (x8.nonrigid_stay > 0.5) & (x8.cross_b2b_backbone > x8.cross_b2b_nonrigid)).all()))

**Result: reproduced, on the assumption that units are linked by their identical ordered
positions.** All 90 HO2 units and all 116 HO3 units at 0 h are linked (HO3's 4 h unit 24 has no
0 h counterpart). Of the 0 h backbone units, 8 of 10 (HO2) and 13 of 13 (HO3) are backbone at 4 h;
of the non-rigid units, 74 of 80 and 97 of 103 stay non-rigid. Across the two recordings, backbone
units have higher burst-to-burst correlations than non-rigid units in both organoids (HO2 0.748
against 0.627, Welch p = 0.039; HO3 0.820 against 0.607, p = 4e-8, 30% rule); with the code's 70%
rule both means are still higher (HO2 0.738 against 0.699, p = 0.62; HO3 p = 0.001). Within each
recording on its own, backbone units are above non-rigid units as well.

## ed5-shuffle-reduces-b2b

> Randomizing spike times while keeping unit and population rates reduces burst-to-burst
> correlations.

In [ ]:
u8 = unit_table(HO8)
d = u8[(u8.bb == 1) & np.isfinite(u8.b2b30) & np.isfinite(u8.b2b_shuf)]
long = pd.concat([d.assign(y=d.b2b30, real=1), d.assign(y=d.b2b_shuf, real=0)])
fa = sq.lmm(long, "y ~ real", "rec")
fa_unit = sq.lmm(long, "y ~ real", "unit")
print(f"Backbone units with both values: {len(d)}; real {d.b2b30.mean():.3f}, shuffled mean {d.b2b_shuf.mean():.3f}")
print(f"(a) real minus shuffled, random intercept per organoid: {fa.params['real']:.3f}, p {fa.pvalues['real']:.2g}; "
      f"per unit: {fa_unit.params['real']:.3f}, p {fa_unit.pvalues['real']:.2g}")
d = d.assign(norm=sq.normalize(d.b2b30, d.b2b_shuf))
fb = sq.lmm(d, "norm ~ 1", "rec")
print(f"(b) normalised mean {fb.params['Intercept']:.3f}, one-sided p {fb.pvalues['Intercept'] / 2 if fb.params['Intercept'] > 0 else 1 - fb.pvalues['Intercept'] / 2:.2g}")
print("Per organoid, real minus shuffled (backbone):", d.groupby("rec").apply(lambda g: round((g.b2b30 - g.b2b_shuf).mean(), 3)).to_dict())
dn = u8[(u8.bb == 0) & np.isfinite(u8.b2b30) & np.isfinite(u8.b2b_shuf)]
print(f"Non-rigid units: real {dn.b2b30.mean():.3f}, shuffled {dn.b2b_shuf.mean():.3f}")

**Result: reproduced.** For backbone units of HO1 to HO8 the real burst-to-burst correlation
(0.605) exceeds the mean over shuffles (0.492): by 0.112 with a random intercept per organoid (p =
1e-26) or per unit (p = 2e-28), and the normalised value is above 0 (0.082, one-sided p = 0.014).
HO8 goes the other way (-0.034); HO7 has no backbone unit. The claims file asks for random
intercepts for unit and organoid together; they are fitted one at a time here.

## fig3e-pairwise-correlation

> Pairwise cross-correlations of firing rates are higher between backbone units than between
> backbone and non-rigid units or between non-rigid units.

In [ ]:
def pair_table(keys, bbkey="bb", src=None):
    src = R if src is None else src
    out = []
    for k in keys:
        r = src[k]
        if "pw_corr" not in r:
            continue
        idx = r["idx30"]
        bb = r[bbkey][idx]
        iu = np.triu_indices(len(idx), 1)
        t = np.where(bb[iu[0]] & bb[iu[1]], "bb", np.where(bb[iu[0]] | bb[iu[1]], "mix", "nr"))
        sh = r.get("shuf", {})
        Bc = np.nanmean(np.array(sh["pw_corr"]), 0)[iu] if sh.get("pw_corr") else np.nan
        Bl = np.nanmean(np.array(sh["pw_lag"]), 0)[iu] if sh.get("pw_lag") else np.nan
        out.append(pd.DataFrame(dict(rec=k, mtype=r["mtype"], type=t, y=r["pw_corr"][iu],
                                     lag=np.abs(r["pw_lag"][iu]), shuf=Bc, shuf_lag=Bl)))
    return pd.concat(out, ignore_index=True)


def type_contrasts(d, group="rec", y="y"):
    f = sq.lmm(d.rename(columns={y: "yy"}), "yy ~ 0 + C(type)", group)
    out = {}
    for other in ["mix", "nr"]:
        if f"C(type)[{other}]" in f.fe_params.index:
            c = sq.contrast(f, {"C(type)[bb]": 1.0, f"C(type)[{other}]": -1.0})
            out[f"bb-{other}"] = (round(c["estimate"], 3), float(f"{c['p']:.2g}"))
    return out


pt = pair_table(HO4)
d = pt[np.isfinite(pt.y)]
print(f"{len(d)} pairs among units with at least 30 spikes; mean by type:", d.groupby("type").y.mean().round(3).to_dict())
print("Contrasts (estimate, p):", type_contrasts(d))
print("Per organoid:", d.groupby(["rec", "type"]).y.mean().round(3).unstack().to_dict("index"))
n_all = sum(R[k]["U"] * (R[k]["U"] - 1) // 2 for k in HO4)
print(f"All unit pairs of HO1 to HO4 (every sorted unit): {n_all} (expected 34056 +- 3406)")

**Result: reproduced.** Backbone pairs (0.70) exceed mixed (0.36) and non-rigid pairs (0.23),
both contrasts with p below 1e-20, in each organoid. The paper's 34,056 pairs is the number of
pairs among all sorted units of HO1 to HO4 (34,056 here as well); 9,648 pairs among units with
at least 30 spikes enter the test.

## fig3f-backbone-pairs-in-tail

> The distribution of pairwise correlations is skewed and roughly lognormal, and backbone-backbone
> pairs lie in its upper tail in every organoid.

In [ ]:
rows = []
for k in HO4:
    x = d[d.rec == k]
    m = x[x.type == "bb"].y.mean()
    lx = np.log(x.y[x.y > 0])
    h, e = np.histogram(lx, bins=30)
    cen = (e[:-1] + e[1:]) / 2
    yh = stats.norm.pdf(cen, lx.mean(), lx.std()) * len(lx) * np.diff(e)[0]
    rows.append(dict(rec=k, pairs=len(x), bb_mean=m, p75=np.percentile(x.y, 75), p90=np.percentile(x.y, 90),
                     percentile_of_bb_mean=stats.percentileofscore(x.y, m), skew=stats.skew(x.y),
                     r2_normal_on_log=1 - ((h - yh) ** 2).sum() / ((h - h.mean()) ** 2).sum()))
f3f = pd.DataFrame(rows)
print(f3f.round(3).to_string(index=False))
print("Above 75th percentile in all:", bool((f3f.bb_mean > f3f.p75).all()), "; above 90th in all:", bool((f3f.bb_mean > f3f.p90).all()))

**Result: partially reproduced.** The backbone-pair mean lies above the 75th percentile of all
pairs in every organoid (at the 87th to 95th percentile), and above the 90th percentile in three
of four (HO3 at the 87th). The distributions are right-skewed (skew 0.26 to 1.08).

## Burst similarity (fig4c, fig4d, fig4e, supp13)

In [ ]:
def bb_window_frames(r):
    lo, hi = sq.backbone_window(r["P"], r["bb"])
    return lo, hi


rows = []
for k in HO8:
    r = R[k]
    if r.get("sim_bb") is None:
        continue
    lo, hi = bb_window_frames(r)
    fr = sq.SIM_FRAMES
    w = (fr >= lo) & (fr <= hi)
    rows.append(dict(rec=k, window=f"{lo} to {hi} ms", bursts=r["n_bursts"],
                     all=np.nanmean(r["sim_all"][w]), backbone=np.nanmean(r["sim_bb"][w]),
                     nonrigid=np.nanmean(r["sim_nr"][w]), shuffled=np.nanmean(r["shuf"]["sim_all"][w])))
print("Mean similarity over the backbone window:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("HO7 has no backbone unit, so no backbone window.")

## fig4c-similarity-variance

> The spread of burst similarity across pairs of bursts is high outside bursts, drops at burst
> onset and stays low until the bursts end.

In [ ]:
rows = []
for k in HO8:
    r = R[k]
    b = r["bursts"]
    fr = sq.SIM_FRAMES
    rel_s, rel_e = b["start"] - b["peak"], b["end"] - b["peak"]
    span = (fr >= max(rel_s.min(), fr[0])) & (fr <= min(rel_e.max(), fr[-1]))
    sd = np.nanstd(r["sim_all"], 1)
    readings = {"a_median_burst": (np.median(rel_s), np.median(rel_e))}
    if r["bb"].sum():
        readings["b_backbone_window"] = bb_window_frames(r)
    for name, (lo, hi) in readings.items():
        inside = span & (fr >= lo) & (fr <= hi)
        outside = span & ~inside
        rows.append(dict(rec=k, reading=name, inside=f"{lo:.0f} to {hi:.0f}", sd_inside=sd[inside].mean(),
                         sd_outside=sd[outside].mean() if outside.any() else np.nan, frames_outside=int(outside.sum())))
f4c = pd.DataFrame(rows)
print(f4c.round(3).to_string(index=False))
ex = f4c[f4c.rec == EXAMPLE]
print("\nExample organoid, inside lower than outside under both readings:", bool((ex.sd_inside < ex.sd_outside).all()))

**Result: reproduced.** In the example organoid HO1 the standard deviation of burst similarity
over burst pairs is lower inside the burst than outside it, under both readings (0.142 against
0.191 with the median burst edges, 0.133 against 0.163 with the backbone window); the same holds
in every organoid shown.

## fig4d-burst-similarity

> Across pairs of bursts, the population firing vectors of backbone units are more similar
> throughout the backbone window than those of non-rigid units or of shuffled data.

In [ ]:
rows = []
for k in HO8:
    r = R[k]
    if r.get("sim_bb") is None:
        continue
    lo, hi = bb_window_frames(r)
    fr = sq.SIM_FRAMES
    w = np.nonzero((fr >= lo) & (fr <= hi))[0]
    alpha = 0.05 / len(w)
    res_k = {}
    for other, X in [("nonrigid", r["sim_nr"]), ("shuffled", r["shuf"]["sim_all"])]:
        ok = 0
        for i in w:
            a, c = r["sim_bb"][i], X[i]
            m = np.isfinite(a) & np.isfinite(c)
            t, p = stats.ttest_rel(a[m], c[m])
            ok += int(t > 0 and p < alpha)
        res_k[other] = ok / len(w)
    rows.append(dict(rec=k, frames=len(w), frac_frames_bb_gt_nonrigid=res_k["nonrigid"],
                     frac_frames_bb_gt_shuffled=res_k["shuffled"]))
f4d = pd.DataFrame(rows)
print(f4d.round(3).to_string(index=False))

**Result: reproduced.** In HO1, backbone similarity is significantly higher than non-rigid and
than shuffled similarity at every one of the 226 frames of the backbone window (Bonferroni). The
same holds at every frame in HO2 to HO6; in HO8 at 67% and 45% of frames.

## fig4e-similarity-rises-at-backbone-onset

> In every human organoid the burst similarity rises from the onset of the backbone period and
> stays high during it.

In [ ]:
rows = []
for k in HO8:
    r = R[k]
    if not r["bb"].sum():
        rows.append(dict(rec=k, note="no backbone unit, no backbone period"))
        continue
    lo, hi = bb_window_frames(r)
    fr = sq.SIM_FRAMES
    m = np.nanmean(r["sim_all"], 1)
    w = (fr >= lo) & (fr <= hi)
    first_val = m[np.nonzero(fr == lo)[0][0]]
    rows.append(dict(rec=k, window=f"{lo} to {hi}", at_onset=first_val, mean_in_period=m[w].mean(),
                     rises=bool(m[w].mean() > first_val)))
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: partially reproduced.** In the seven organoids that have backbone units, the mean
similarity over the backbone period exceeds its value at the period's first frame (in HO8 barely,
0.334 against 0.323). HO7 has no backbone unit, so no backbone period, and the claim holds in 7 of
8 organoids.

## supp13-top-correlated-units-similarity

> The 20% of units with the highest average pairwise correlation give a higher burst similarity
> than all units together, and the 20% with the lowest give a lower one.

In [ ]:
r = R[EXAMPLE]
m_ex = sq.load_matrix(first.loc[EXAMPLE].asset_id, sq.keep_seconds(EXAMPLE))
Z = sq.zscore_rows(sq.isi_rates(m_ex))
idx = r["idx30"]
C = r["pw_corr"].copy()
C = np.where(np.isnan(C), C.T, C)
np.fill_diagonal(C, np.nan)
avg = np.nanmean(C, 1)
order = idx[np.argsort(avg)]  # ascending mean pairwise correlation
lo, hi = bb_window_frames(r)
fr = sq.SIM_FRAMES
w = np.nonzero((fr >= lo) & (fr <= hi))[0]
peaks = r["bursts"]["peak"]
sim_all = r["sim_all"][w]


def subset_sim(units):
    return sq.burst_similarity(Z[units], peaks, fr[w])


n20 = int(round(0.2 * len(order)))
top, bot = subset_sim(order[-n20:]), subset_sim(order[:n20])
alpha = 0.05 / len(w)
res13 = {}
for name, X, sign in [("top20_higher", top, 1), ("bottom20_lower", bot, -1)]:
    ok = 0
    for i in range(len(w)):
        mm = np.isfinite(X[i]) & np.isfinite(sim_all[i])
        t, p = stats.ttest_rel(X[i][mm], sim_all[i][mm])
        ok += int(sign * t > 0 and p < alpha)
    res13[name] = ok / len(w)
print(f"{EXAMPLE}: {len(order)} units with at least 30 spikes, 20% = {n20}; backbone window {lo} to {hi} ms ({len(w)} frames)")
print("Mean similarity in the window: all", round(np.nanmean(sim_all), 3), "top 20%", round(np.nanmean(top), 3),
      "bottom 20%", round(np.nanmean(bot), 3))
print("Fraction of window frames significant (Bonferroni):", {k: round(v, 3) for k, v in res13.items()})
diffs = {}
for pct in [20, 40, 60, 80, 95]:
    n = max(1, int(round(pct / 100 * len(order))))
    diffs[pct] = round(float(np.nansum(np.nanmean(subset_sim(order[-n:]), 1)) - np.nansum(np.nanmean(subset_sim(order[:n]), 1))), 2)
print("Summed similarity over the window, top minus bottom, by percentile:", diffs)
del Z, m_ex

**Result: reproduced.** In HO1 the top 20% of units by mean pairwise correlation (19 units) give
a higher burst similarity than all units (0.868 against 0.453) and the bottom 20% a lower one
(0.056), each significant at every frame of the backbone window; the top-minus-bottom difference
in summed similarity shrinks from 183.6 at 20% to 6.3 at 95%.

## fig4f-pca-variance

> In the example organoid, the first two principal components of backbone unit rates explain 73%
> of their variance against 25% for non-rigid units, and across organoids backbone units need
> fewer components than all units while non-rigid units need more.

In [ ]:
rows = []
for k in HO8:
    p = R[k]["pca"]
    rows.append(dict(rec=k, **{f"{s}_2pc": (100 * p[s][:2].sum() if p[s] is not None else np.nan) for s in ["all", "bb", "nr"]}))
f4f = pd.DataFrame(rows)
f4f["bb_gt_all"] = f4f.bb_2pc > f4f.all_2pc
f4f["nr_lt_all"] = f4f.nr_2pc < f4f.all_2pc
print(f4f.round(1).to_string(index=False))
print("Expected for the example organoid: backbone 73 +- 7.3, non-rigid 25 +- 2.5 percent")
match = f4f[(f4f.bb_2pc.between(65.7, 80.3)) & (f4f.nr_2pc.between(22.5, 27.5))]
print("Organoids matching both:", match.rec.tolist())
rows = []
for k in HO8:
    p = R[k]["pca_code"]
    rows.append(dict(rec=k, **{f"{s}_2pc": (100 * p[s][:2].sum() if p[s] is not None else np.nan) for s in ["all", "bb", "nr"]}))
f4c_code = pd.DataFrame(rows)
print("\nVariant, the code's burst-only rates (inverse ISI held across the gaps between bursts):")
print(f4c_code.round(1).to_string(index=False))
print("Organoids matching both:", f4c_code[(f4c_code.bb_2pc.between(65.7, 80.3)) & (f4c_code.nr_2pc.between(22.5, 27.5))].rec.tolist())

**Result: partially reproduced.** In HO1 the first two PCs explain 75.4% of the backbone units'
variance (within 73 ± 7.3) but 17.4% for non-rigid units (outside 25 ± 2.5); no organoid matches
both values. With the code's burst-only rates, which hold the inverse ISI across the gaps between
bursts, HO1 gives 75.1% and 15.3%, so the zeroing between bursts does not explain the difference.
In each of the seven organoids with backbone units, backbone units have a higher and non-rigid
units a lower two-PC variance than all units; HO7 has no backbone unit.

## Neonatal slices (fig6f, fig6g)

In [ ]:
MS = ["M1S1", "M1S2", "M2S1", "M2S2", "M3S1", "M3S2"]
for key, lab in [("bb", "90% (Methods)"), ("bb_70", "70% (Fig. 6B legend)")]:
    us = unit_table(MS, key)
    us["animal"] = us.rec.str[:2]
    d = us[np.isfinite(us.b2b30)]
    for g in ["rec", "animal"]:
        f = sq.lmm(d.rename(columns={"b2b30": "y"}), "y ~ bb", g)
        print(f"{lab}, random intercept per {g}: {len(d)} units ({int(d.bb.sum())} backbone); backbone {d[d.bb == 1].b2b30.mean():.3f}, "
              f"non-rigid {d[d.bb == 0].b2b30.mean():.3f}; effect {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")
print("Per slice (90%):", unit_table(MS).dropna(subset=["b2b30"]).groupby(["rec", "bb"]).b2b30.mean().round(3).unstack().to_dict("index"))
print(f"All sorted units in the six slices: {sum(R[k]['U'] for k in MS)} (expected 959 +- 96)")

**Result: reproduced.** Backbone units have higher burst-to-burst correlations under both
thresholds and both random-effect structures (effects 0.122 to 0.159, p from 5e-17 to 2e-37), and
under the second reading of the recording length (p below 1e-33, table under "Recording length,
reading (b)"). The paper's n of 959 is the number of all sorted units in the six slices (959
here); 685 pass the unit rules.

In [ ]:
for key, lab in [("bb", "90%"), ("bb_70", "70%")]:
    ps = pair_table(MS, key)
    ps["animal"] = ps.rec.str[:2]
    d = ps[np.isfinite(ps.y)]
    print(f"{lab}: {len(d)} pairs; means {d.groupby('type').y.mean().round(3).to_dict()}")
    for g in ["rec", "animal"]:
        print(f"   random intercept per {g}: contrasts {type_contrasts(d, g)}")
n_all = sum(R[k]["U"] * (R[k]["U"] - 1) // 2 for k in MS)
print(f"All unit pairs in the six slices: {n_all} (expected 125607 +- 12561)")

**Result: reproduced.** Backbone pairs exceed mixed and non-rigid pairs under every reading,
both recording lengths included (contrasts 0.16 to 0.26, p printed as 0). The paper's 125,607
pairs is the number of pairs among all sorted units of the slices (125,607 here); 89,669 pairs
among units with at least 30 spikes enter the test.

## Comparisons across model types (Fig. 7)

Recordings without bursts (MO2, MO4, MO5, MO6) have no backbone units and are left out; MO3
(four bursts) is kept; the judged murine contrasts without MO3 are shown after the second
reading of the recording length. Slices are shown at both backbone thresholds.

In [ ]:
ALL = [k for k in RECS if R[k]["n_bursts"] >= 2]


def unit_all(slice_key="bb"):
    parts = []
    for k in ALL:
        key = slice_key if R[k]["mtype"] == "MS" else "bb"
        parts.append(unit_table([k], key))
    return pd.concat(parts, ignore_index=True)


def within_type(d, y, types=("HO", "MO", "MS", "PR")):
    d = d.copy()
    d["cell"] = d.mtype + "_" + np.where(d.bb == 1, "bb", "nr")
    f = sq.lmm(d.rename(columns={y: "yy"}), "yy ~ 0 + C(cell)", "rec")
    return f, {t: sq.cell_means_contrast(f, f"{t}_bb", f"{t}_nr") for t in types}


print("Recordings in the Fig. 7 comparisons:", len(ALL), ALL)

### fig7a-backbone-higher-rate and fig7a-primary-higher-rate

In [ ]:
f7a = {}
for sk in ["bb", "bb_70"]:
    ua = unit_all(sk)
    d = ua[ua.n_spk >= sq.MIN_SPIKES].assign(lr=lambda x: np.log10(x.rate))
    f, con = within_type(d, "lr")
    f7a[sk] = (d, f)
    print(f"Slices at {'90' if sk == 'bb' else '70'}%: units per type {d.groupby('mtype').size().to_dict()}")
    for t, c in con.items():
        print(f"   {t}: backbone minus non-rigid log10 rate {c['estimate']:.3f}, p {c['p']:.2g}")
print("Expected units: HO 1048, MO 1179, MS 786, PR 1048")
print("All sorted units by type (main recordings):",
      pd.Series({k: R[k]["U"] for k in RECS}).groupby(lambda k: R[k]["mtype"]).sum().to_dict())

**Result: reproduced.** Backbone units fire faster than non-rigid units in every model type
(log10 rate differences 0.34 to 0.81, p below 1e-28), at both slice thresholds and both recording
lengths. The unit counts differ from the legend's: 699, 624, 813 and 1744 units with at least 30
spikes (HO, MO, MS, PR) against 1048, 1179, 786 and 1048; the four murine organoids without bursts
are left out. The dandiset holds 930 sorted human organoid units in the main recordings, fewer than
the legend's 1048.

In [ ]:
for sk in ["bb", "bb_70"]:
    d, f = f7a[sk]
    out = {}
    for g in ["bb", "nr"]:
        for t in ["HO", "MO", "MS"]:
            c = sq.cell_means_contrast(f, f"PR_{g}", f"{t}_{g}")
            out[f"PR-{t} {g}"] = (round(c["estimate"], 3), float(f"{c['p']:.2g}"))
    print(f"Slices at {'90' if sk == 'bb' else '70'}%:", out)
    print("   favouring primaries with p < 0.05:", sum(v[0] > 0 and v[1] < 0.05 for v in out.values()), "of 6")
print("Median rate by type and unit group:", d.groupby(["mtype", "bb"]).rate.median().round(2).unstack().to_dict("index"))

**Result: partially reproduced.** Five of the six contrasts favour primary cultures with p <
0.05; primary backbone units do not fire faster than murine organoid backbone units (difference
0.04 log10 units, p = 0.66 and 0.67; 0.13, p = 0.20 under the second recording length). Without MO3 the primary against murine organoid backbone contrast is -0.04, p = 0.69. The murine organoid results rest on five of nine recordings, with burst detection that the paper does not give for murine organoids (data check), and may come from that difference.

### fig7b-normalized-b2b and fig7b-primary-no-b2b-difference

In [ ]:
f7b = {}
for sk in ["bb", "bb_70"]:
    ua = unit_all(sk)
    for norm in ["ratio", "difference"]:
        d = ua[np.isfinite(ua.b2b30) & np.isfinite(ua.b2b_shuf)].copy()
        d["nb"] = sq.normalize(d.b2b30, d.b2b_shuf) if norm == "ratio" else d.b2b30 - d.b2b_shuf
        f, con = within_type(d, "nb")
        f7b[(sk, norm)] = (d, f, con)
        cross = {t: sq.cell_means_contrast(f, f"{t}_bb", "PR_bb") for t in ["HO", "MO", "MS"]}
        print(f"Slices {'90' if sk == 'bb' else '70'}%, {norm}: units {len(d)}")
        print("   backbone minus non-rigid:", {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})
        print("   backbone minus primary backbone:", {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in cross.items()})

**Result: not reproduced.** With the ratio normalisation, backbone units have higher
burst-to-burst correlations than non-rigid units in human organoids (0.083, p = 2e-10) and slices
(p below 1e-15), but not in murine organoids (0.026, p = 0.11; -0.003, p = 0.84 under the second
recording length). Backbone units of human organoids and slices exceed primary-culture backbone
units at the 90% slice threshold (p = 0.043 and 0.043), murine organoid ones do not (p = 0.59), and
the slice contrast is not significant at 70% (p = 0.11). The criterion's partial outcome needs the
three within-type contrasts, and the murine one fails. With the difference normalisation (a
variant), murine organoids are above (0.044, p = 0.007). Without MO3 the murine organoid contrast holds (0.075, p = 1e-4), which would meet the criterion's partial outcome, so MO3, with four bursts under the stated detection, decides this outcome. The murine organoid results rest on five of nine recordings, with burst detection that the paper does not give for murine organoids (data check), and may come from that difference.

In [ ]:
for (sk, norm), (d, f, con) in f7b.items():
    c = con["PR"]
    print(f"Slices {'90' if sk == 'bb' else '70'}%, {norm}: primary backbone minus non-rigid {c['estimate']:.3f}, p {c['p']:.2g}")

**Result: reproduced.** In primary cultures normalised burst-to-burst correlations do not differ
between backbone and non-rigid units (-0.002, p = 0.81; difference normalisation p = 0.74; p =
0.98 under the second recording length).

### fig7c (normalized pairwise correlations)

In [ ]:
pair_parts = []
for sk in ["bb", "bb_70"]:
    for k in ALL:
        key = sk if R[k]["mtype"] == "MS" else "bb"
        p = pair_table([k], key)
        p["slice_key"] = sk
        pair_parts.append(p)
pa = pd.concat(pair_parts, ignore_index=True)
pa = pa[np.isfinite(pa.y) & np.isfinite(pa.shuf)]
pa["norm"] = sq.normalize(pa.y, pa.shuf)
pa["diff"] = pa.y - pa.shuf
print(pa[pa.slice_key == "bb"].groupby(["mtype", "type"]).norm.agg(["count", "median"]).round(3).unstack().to_string())


def median_model(sk, v, types=("HO", "MO", "MS", "PR"), pair_level=False):
    d = pa[(pa.slice_key == sk) & pa.type.isin(["bb", "nr"])]
    if not pair_level:
        d = d.groupby(["rec", "mtype", "type"])[v].median().reset_index()
    d = d.assign(cell=d.mtype + "_" + d.type)
    f = sq.lmm(d.rename(columns={v: "yy"}), "yy ~ 0 + C(cell)", "rec")
    names = set(f.fe_params.index)
    return {t: sq.cell_means_contrast(f, f"{t}_bb", f"{t}_nr") for t in types
            if f"C(cell)[{t}_bb]" in names and f"C(cell)[{t}_nr]" in names}


f7c = {}
for sk in ["bb", "bb_70"]:
    for v in ["norm", "diff"]:
        for pl in [False, True]:
            f7c[(sk, v, pl)] = median_model(sk, v, pair_level=pl)

In [ ]:
for (sk, v, pl), con in f7c.items():
    print(f"slices {'90' if sk == 'bb' else '70'}%, {v}, {'pair level' if pl else 'per-recording medians'}:",
          {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})

**Result: reproduced.** On per-recording medians of the ratio-normalised correlations,
backbone pairs exceed non-rigid pairs in human organoids (0.163, p = 0.001) and murine organoids
(0.236, p = 2e-4, five recordings with bursts), at both slice thresholds and both recording
lengths; the difference normalisation and the pair-level variant agree.

In [ ]:
for (sk, v, pl), con in f7c.items():
    if v == "norm" and not pl:
        print(f"slices {'90' if sk == 'bb' else '70'}%: MS {con['MS']['estimate']:.3f} p {con['MS']['p']:.2g}; "
              f"PR {con['PR']['estimate']:.3f} p {con['PR']['p']:.2g}")

**Result: reproduced.** On the same model, backbone and non-rigid pairs do not differ in slices
(0.053, p = 0.29; 0.068, p = 0.20 at 70%) or primary cultures (0.031, p = 0.47), nor under the
second recording length (p = 0.24 to 0.51). The pair-level variant finds both significant (p below
1e-25).

In [ ]:
for sk in ["bb", "bb_70"]:
    out = {}
    for t in ["HO", "MO", "MS", "PR"]:
        d = pa[(pa.slice_key == sk) & (pa.mtype == t) & (pa.type == "bb")]
        f = sq.lmm(d, "norm ~ 1", "rec")
        est, p2 = f.params["Intercept"], f.pvalues["Intercept"]
        p1 = p2 / 2 if est > 0 else 1 - p2 / 2
        out[t] = dict(pairs=len(d), recordings=d.rec.nunique(), mean=round(est, 4), p_one_sided=float(f"{p1:.2g}"),
                      p_two_sided=float(f"{p2:.2g}"))
    print(f"slices {'90' if sk == 'bb' else '70'}%:")
    print(pd.DataFrame(out).T.to_string())

**Result: not reproduced.** Normalised backbone pair correlations are above 0 in human
organoids (0.164, one-sided p = 6e-4) but not in murine organoids (0.008, p = 0.36), and not in
slices (0.076, p = 0.065 at 90%; 0.070, p = 0.091 at 70%; under the second recording length p =
0.10 and 0.038). In primary cultures they are above 0 (0.043, p = 0.017), where the paper finds no
effect. Without MO3 murine organoid backbone pairs are still not above 0 (0.020, p = 0.23). The murine organoid results rest on five of nine recordings, with burst detection that the paper does not give for murine organoids (data check), and may come from that difference.

### fig7d lag times

In [ ]:
lagd = pa[(pa.slice_key == "bb") & (pa.type == "bb")].copy()
print(lagd.groupby("mtype").lag.describe().round(1).to_string())
f7d = {}
for reading in ["a_drop_zero", "b_zero_as_1ms"]:
    d = lagd[lagd.lag > 0] if reading == "a_drop_zero" else lagd.assign(lag=lagd.lag.clip(lower=1))
    d = d.assign(ll=np.log10(d.lag))
    f = sq.lmm(d, "ll ~ 0 + C(mtype)", "rec")
    f7d[reading] = {f"{a}-{b}": sq.contrast(f, {f"C(mtype)[{a}]": 1.0, f"C(mtype)[{b}]": -1.0})
                    for a, b in [("HO", "PR"), ("MO", "PR"), ("HO", "MO"), ("HO", "MS"), ("MO", "MS"), ("MS", "PR")]}
    print(f"{reading}: {len(d)} backbone pairs; log10 lag means", d.groupby("mtype").ll.mean().round(3).to_dict())
    print("   ", {k: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for k, c in f7d[reading].items()})

In [ ]:
for reading, con in f7d.items():
    print(reading, {k: (round(con[k]["estimate"], 3), float(f"{con[k]['p']:.2g}")) for k in ["HO-PR", "MO-PR", "HO-MO", "HO-MS"]})

**Result: partially reproduced.** Backbone lags are longer in human organoids (0.86 log10 units,
p = 7e-7) and murine organoids (0.70, p = 4e-5) than in primary cultures, and human organoids
differ from slices (p = 0.002), but not from murine organoids (0.15, p = 0.29). Setting zero lags
to 1 ms, and the second recording length, give the same pattern. The code's lag table
(Processed_data/Figure7/lag_data.xlsx) has no zero lag, so reading (a), dropping them, is the
code's. Without MO3 human and murine organoid lags still do not differ (0.23, p = 0.13). The murine organoid results rest on five of nine recordings, with burst detection that the paper does not give for murine organoids (data check), and may come from that difference.

In [ ]:
for reading, con in f7d.items():
    print(reading, {k: (round(con[k]["estimate"], 3), float(f"{con[k]['p']:.2g}")) for k in ["MO-MS", "MS-PR"]})

**Result: not reproduced, on the assumption that zero lags are dropped as in the code.** Under
reading (a), the code's, murine organoid lags are longer than slice lags (0.31 log10 units, p =
0.045); with zero lags set to 1 ms, p = 0.07; under the second recording length (zero lags
dropped), p = 0.013. The second contrast, slices against primary cultures, differs under every
reading (p = 0.008, 0.006 and 0.029), where the paper reports P = 0.53. Without MO3 murine organoid and slice lags do not differ (0.24, p = 0.14), as the claim expects; the slices against primary cultures contrast does not involve murine organoids, and the outcome does not change. The murine organoid results rest on five of nine recordings, with burst detection that the paper does not give for murine organoids (data check), and may come from that difference.

### fig7e-pca-backbone-variance

In [ ]:
rows = []
for k in ALL:
    r = R[k]
    p = r["pca"]
    if any(p[s] is None for s in ["all", "bb", "nr"]):
        continue
    A = sq.pca_ratio(p["all"], p["bb"], p["nr"])
    Bs = [sq.pca_ratio(x["all"], x["bb"], x["nr"]) for x in r["shuf"]["pca"]
          if all(x[s] is not None for s in ["all", "bb", "nr"])]
    B = np.mean(Bs, 0)
    for i, s in enumerate(["all", "bb", "nr"]):
        rows.append(dict(rec=k, mtype=r["mtype"], sel=s, value=A[i], shuf=B[i], norm=sq.normalize(A[i], B[i]), diff=A[i] - B[i]))
pc = pd.DataFrame(rows)
print(pc.pivot_table(index=["mtype", "rec"], columns="sel", values="norm").round(3).to_string())
for v in ["norm", "diff"]:
    for reading, other in [("a", "nr"), ("b", "all")]:
        d = pc[pc.sel.isin(["bb", other])] if reading == "a" else pc
        d = d.assign(cell=d.mtype + "_" + d.sel)
        f = sq.lmm(d.rename(columns={v: "yy"}), "yy ~ 0 + C(cell)", "rec")
        con = {t: sq.cell_means_contrast(f, f"{t}_bb", f"{t}_{other}") for t in ["HO", "MO", "MS", "PR"]}
        print(f"{v}, reading ({reading}) backbone minus {other}:", {t: (round(c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})
d = pc[pc.sel == "bb"]
f = sq.lmm(d, "norm ~ 0 + C(mtype)", "rec")
print("Backbone, organoids against other types:",
      {f"{a}-{b}": float(f"{sq.contrast(f, {f'C(mtype)[{a}]': 1.0, f'C(mtype)[{b}]': -1.0})['p']:.2g}") for a in ["HO", "MO"] for b in ["MS", "PR"]})

**Result: not reproduced.** The normalised three-PC variance of backbone units exceeds that of
non-rigid units in human organoids (0.188, p = 1e-6) but not in murine organoids (-0.040, p =
0.25); against all units, human organoids again (p = 1e-6) and murine organoids not (p = 0.24),
and the same under the second recording length. Human organoid backbone units exceed those of
slices (p = 0.014) and primary cultures (p = 8e-4); murine organoid ones do not. Without MO3 murine organoids are still not above (-0.064, p = 0.095; 0.034, p = 0.23). The murine organoid results rest on five of nine recordings, with burst detection that the paper does not give for murine organoids (data check), and may come from that difference.

### supp24c-nonrigid-more-variable

In [ ]:
for sk in ["bb", "bb_70"]:
    ua = unit_all(sk)
    d = ua[(ua.n_spk >= sq.MIN_SPIKES) & np.isfinite(ua.cv2)].copy()
    d["cv2z"] = (d.cv2 - d.cv2_shuf_mean) / d.cv2_shuf_sd
    for v in ["cv2", "cv2z"]:
        dd = d[np.isfinite(d[v])]
        f, con = within_type(dd, v)
        print(f"slices {'90' if sk == 'bb' else '70'}%, {v}: non-rigid minus backbone",
              {t: (round(-c["estimate"], 3), float(f"{c['p']:.2g}")) for t, c in con.items()})

**Result: reproduced.** Non-rigid units have higher CV2 than backbone units in all four model
types, raw (p from 1e-6 to 5e-39) and z-scored to the shuffles (p from 0.018 to 3e-53), at both
slice thresholds and both recording lengths.

### supp10b-rank-order-by-model

In [ ]:
rk_all = rank_table(R, ALL)
rk_all["mtype"] = rk_all.rec.map(lambda k: R[k]["mtype"])
print(rk_all.groupby("mtype")[["raw", "norm", "z"]].median().round(3).to_string())
for v in ["norm", "z"]:
    d = rk_all[np.isfinite(rk_all[v])]
    f = sq.lmm(d, f"{v} ~ 0 + C(mtype)", "rec")
    names = list(f.fe_params.index)
    Wm = np.array([[1, -1, 0, 0], [1, 0, -1, 0], [1, 0, 0, -1]], float)
    cov = f.cov_params().loc[names, names].values
    est = Wm @ f.fe_params.values
    chi2 = float(est @ np.linalg.solve(Wm @ cov @ Wm.T, est))
    df_min = min(f.satterthwaite(w) for w in Wm)
    c = sq.contrast(f, {"C(mtype)[HO]": 1.0, "C(mtype)[PR]": -1.0})
    print(f"{v}: {len(d)} burst pairs in {d.rec.nunique()} recordings; main effect F {chi2 / 3:.2f} on 3 and {df_min:.1f} df "
          f"(the smallest Satterthwaite df of the three contrasts), p {stats.f.sf(chi2 / 3, 3, df_min):.2g} (Wald chi2(3) p {stats.chi2.sf(chi2, 3):.2g}); "
          f"HO minus PR {c['estimate']:.3f}, t {c['t']:.2f} on {c['df']:.1f} df, p {c['p']:.2g}")

**Result: recorded as reproduced, on the assumptions that the z-score normalisation and the
code's recording lengths are the ones used; a change to blocked (methods_underspecified) is
proposed for the third verification round.** With z-scored rank-order correlations the model type
matters (F test p = 0.032) and human organoids exceed primary cultures (1.53, p = 0.031), both
weaker than the paper's P = 0.004 and 0.0004. With the ratio normalisation neither holds (p = 0.66
and 0.49). Under the second recording length the z-scored contrasts are not significant (p = 0.064
and 0.099, with 5 shuffles per recording), and the ratio contrasts are not either (p = 0.69 and
0.38). The criterion requires both normalisation readings, and the claims file runs both recording
lengths where the outcome could depend on them; the claim passes under one of the four
combinations only, so the readings disagree and the claim is underspecified, as for ed4d.

## Recording length, reading (b): the shortest initial segment with 38 to 40 bursts

The claims file names two readings of the recording length: (a) the code's lengths, used above,
and (b) the shortest initial segment of each recording that holds 38 to 40 bursts (searched in 5 s
steps; the whole recording when it holds fewer). Under (b) every recording is analysed again
where its length differs from (a) by more than a second, with 5 shuffles (2 for the pairwise
correlations) and the peak times taken from the rates of the cut recording, to fit the runtime and
memory. The table below gives every judged contrast of Figs. 6 and 7 and Supplementary Figs. 10B
and 24C under both readings.

In [ ]:
def find_length(k):
    pop = sq.load_pop(first.loc[k].asset_id)
    return k, len(pop), sq.length_for_bursts(pop, sq.model_type(k))


with ThreadPoolExecutor(4) as ex:
    lengths = pd.DataFrame([dict(rec=k, total_s=n / 1000, a_code_s=R[k]["T"] / 1000, b_40_bursts_s=m / 1000)
                            for k, n, m in ex.map(find_length, RECS)]).set_index("rec")
redo = lengths.index[(lengths.b_40_bursts_s - lengths.a_code_s).abs() > 1]


def run_b(args):
    k, n, seed = args
    m = sq.load_matrix(first.loc[k].asset_id, n / 1000)
    return k, sq.analyse(k, m, seed, n_shuf=5, n_shuf_pair=2, cut_rates=True)


t0 = time.time()
R40 = dict(R)
with ProcessPoolExecutor(2) as ex:
    for k, r in ex.map(run_b, [(k, int(lengths.loc[k, "b_40_bursts_s"] * 1000), 4000 + i) for i, k in enumerate(redo)]):
        R40[k] = r
lengths["bursts_a"] = [R[k]["n_bursts"] for k in lengths.index]
lengths["bursts_b"] = [R40[k]["n_bursts"] for k in lengths.index]
lengths["backbone_a"] = [int(R[k]["bb"].sum()) if "bb" in R[k] else np.nan for k in lengths.index]
lengths["backbone_b"] = [int(R40[k]["bb"].sum()) if "bb" in R40[k] else np.nan for k in lengths.index]
lengths["reanalysed"] = lengths.index.isin(redo)
print(lengths.to_string())
print(f"{len(redo)} recordings analysed again in {time.time() - t0:.0f} s")

In [ ]:
def judged_contrasts(src, exclude=()):
    """Every judged contrast of Figs. 6 and 7 and Supplementary Figs. 10B and 24C on one set of
    analysed recordings, leaving out the recordings in exclude: (claim, contrast) -> (estimate, p),
    the same models as the cells above."""
    keys = [k for k in RECS if src[k]["n_bursts"] >= 2 and k not in exclude]
    out = {}

    def put(claim, label, c):
        out[(claim, label)] = (c["estimate"], c["p"])

    def bbkey(k, sk):
        return sk if src[k]["mtype"] == "MS" else "bb"

    for sk in ["bb", "bb_70"]:
        tag = "slices 90%" if sk == "bb" else "slices 70%"
        ua = pd.concat([unit_table([k], bbkey(k, sk), src) for k in keys], ignore_index=True)
        d = ua[(ua.mtype == "MS") & np.isfinite(ua.b2b30)]
        f = sq.lmm(d.rename(columns={"b2b30": "y"}), "y ~ bb", "rec")
        put("fig6f-slice-b2b-correlation", f"MS backbone - non-rigid, {tag}", sq.contrast(f, {"bb": 1.0}))
        d = ua[ua.n_spk >= sq.MIN_SPIKES].assign(lr=lambda x: np.log10(x.rate))
        f, con = within_type(d, "lr")
        for t, c in con.items():
            put("fig7a-backbone-higher-rate", f"{t} backbone - non-rigid, {tag}", c)
        for g in ["bb", "nr"]:
            for t in ["HO", "MO", "MS"]:
                put("fig7a-primary-higher-rate", f"PR - {t} {g}, {tag}", sq.cell_means_contrast(f, f"PR_{g}", f"{t}_{g}"))
        d = ua[(ua.n_spk >= sq.MIN_SPIKES) & np.isfinite(ua.cv2)]
        f, con = within_type(d, "cv2")
        for t, c in con.items():
            put("supp24c-nonrigid-more-variable", f"{t} non-rigid - backbone (raw CV2), {tag}", dict(estimate=-c["estimate"], p=c["p"]))
        d = ua[np.isfinite(ua.b2b30) & np.isfinite(ua.b2b_shuf)].copy()
        d["nb"] = sq.normalize(d.b2b30, d.b2b_shuf)
        f, con = within_type(d, "nb")
        for t in ["HO", "MO", "MS"]:
            put("fig7b-normalized-b2b", f"{t} backbone - non-rigid, {tag}", con[t])
            put("fig7b-normalized-b2b", f"{t} backbone - PR backbone, {tag}", sq.cell_means_contrast(f, f"{t}_bb", "PR_bb"))
        put("fig7b-primary-no-b2b-difference", f"PR backbone - non-rigid, {tag}", con["PR"])
        pp = pd.concat([pair_table([k], bbkey(k, sk), src) for k in keys], ignore_index=True)
        pp = pp[np.isfinite(pp.y)]
        ms = pp[pp.mtype == "MS"]
        f = sq.lmm(ms.rename(columns={"y": "yy"}), "yy ~ 0 + C(type)", "rec")
        for other in ["mix", "nr"]:
            put("fig6g-slice-pairwise-correlation", f"MS bb - {other} pairs, {tag}",
                sq.contrast(f, {"C(type)[bb]": 1.0, f"C(type)[{other}]": -1.0}))
        pn = pp[np.isfinite(pp.shuf)].assign(norm=lambda x: sq.normalize(x.y, x.shuf))
        dm = pn[pn.type.isin(["bb", "nr"])].groupby(["rec", "mtype", "type"]).norm.median().reset_index()
        dm = dm.assign(cell=dm.mtype + "_" + dm.type)
        f = sq.lmm(dm.rename(columns={"norm": "yy"}), "yy ~ 0 + C(cell)", "rec")
        names = set(f.fe_params.index)
        for t in ["HO", "MO", "MS", "PR"]:
            if f"C(cell)[{t}_bb]" in names and f"C(cell)[{t}_nr]" in names:
                claim = "fig7c-normalized-pairwise" if t in ("HO", "MO") else "fig7c-slices-primaries-no-pair-difference"
                put(claim, f"{t} bb - nr pairs (recording medians), {tag}", sq.cell_means_contrast(f, f"{t}_bb", f"{t}_nr"))
        for t in ["HO", "MO", "MS", "PR"]:
            d = pn[(pn.mtype == t) & (pn.type == "bb")]
            if d.rec.nunique() >= 2:
                f = sq.lmm(d, "norm ~ 1", "rec")
                est, p2 = f.params["Intercept"], f.pvalues["Intercept"]
                put("fig7c-backbone-pairs-above-random", f"{t} bb pairs above 0 (one-sided), {tag}",
                    dict(estimate=est, p=p2 / 2 if est > 0 else 1 - p2 / 2))
        if sk == "bb":
            lag = pn[(pn.type == "bb") & (pn.lag > 0)].assign(ll=lambda x: np.log10(x.lag))
            f = sq.lmm(lag, "ll ~ 0 + C(mtype)", "rec")
            for a, b in [("HO", "PR"), ("MO", "PR"), ("HO", "MO"), ("HO", "MS")]:
                put("fig7d-lag-times", f"{a} - {b} log10 lag", sq.contrast(f, {f"C(mtype)[{a}]": 1.0, f"C(mtype)[{b}]": -1.0}))
            for a, b in [("MO", "MS"), ("MS", "PR")]:
                put("fig7d-murine-organoids-slices-same-lag", f"{a} - {b} log10 lag",
                    sq.contrast(f, {f"C(mtype)[{a}]": 1.0, f"C(mtype)[{b}]": -1.0}))
    rows = []
    for k in keys:
        p = src[k]["pca"]
        if any(p[s] is None for s in ["all", "bb", "nr"]) or not src[k]["shuf"]["pca"]:
            continue
        A = sq.pca_ratio(p["all"], p["bb"], p["nr"])
        Bs = [sq.pca_ratio(x["all"], x["bb"], x["nr"]) for x in src[k]["shuf"]["pca"]
              if all(x[s] is not None for s in ["all", "bb", "nr"])]
        B = np.mean(Bs, 0)
        rows += [dict(rec=k, mtype=src[k]["mtype"], sel=s, norm=sq.normalize(A[j], B[j])) for j, s in enumerate(["all", "bb", "nr"])]
    pc_ = pd.DataFrame(rows)
    for reading, other in [("a", "nr"), ("b", "all")]:
        d = (pc_[pc_.sel.isin(["bb", "nr"])] if reading == "a" else pc_).assign(cell=lambda x: x.mtype + "_" + x.sel)
        f = sq.lmm(d.rename(columns={"norm": "yy"}), "yy ~ 0 + C(cell)", "rec")
        for t in ["HO", "MO", "MS", "PR"]:
            put("fig7e-pca-backbone-variance", f"{t} backbone - {other} ({reading})", sq.cell_means_contrast(f, f"{t}_bb", f"{t}_{other}"))
    rk_ = rank_table(src, keys)
    rk_["mtype"] = rk_.rec.map(lambda k: src[k]["mtype"])
    for v in ["z", "norm"]:
        d = rk_[np.isfinite(rk_[v])]
        f = sq.lmm(d, f"{v} ~ 0 + C(mtype)", "rec")
        names = list(f.fe_params.index)
        Wm = np.array([[1, -1, 0, 0], [1, 0, -1, 0], [1, 0, 0, -1]], float)
        cov = f.cov_params().loc[names, names].values
        est = Wm @ f.fe_params.values
        F = float(est @ np.linalg.solve(Wm @ cov @ Wm.T, est)) / 3
        dfs = [f.satterthwaite(w) for w in Wm]
        # F on 3 and the smallest Satterthwaite df of the three contrasts (a conservative choice)
        put("supp10b-rank-order-by-model", f"main effect, {v} (F)", dict(estimate=F, p=stats.f.sf(F, 3, min(dfs))))
        put("supp10b-rank-order-by-model", f"HO - PR, {v}", sq.contrast(f, {"C(mtype)[HO]": 1.0, "C(mtype)[PR]": -1.0}))
    return out


ja, jb = judged_contrasts(R), judged_contrasts(R40)
jt = pd.DataFrame({"a_estimate": {k: v[0] for k, v in ja.items()}, "a_p": {k: v[1] for k, v in ja.items()},
                   "b_estimate": {k: v[0] for k, v in jb.items()}, "b_p": {k: v[1] for k, v in jb.items()}})
jt.index.names = ["claim", "contrast"]
pd.set_option("display.max_colwidth", 80)
print(jt.to_string(float_format=lambda v: f"{v:.3g}"))

Under reading (b) the lengths run from 80 s (Pr5) to 1270 s (M3S2), and 25 recordings are
analysed again; backbone sets change most in the slices (M2S1 149 to 123 units, M1S1 31 to 22) and
MO7 (40 to 49). The judged contrasts keep their direction and significance under (b), with these
exceptions: the murine organoid backbone against non-rigid contrast of fig7b changes sign
(0.026 to -0.003, non-significant under both); the z-scored model-type effect of supp10b is no longer significant (main effect p =
0.064, human organoids against primary cultures p = 0.099); slice backbone pairs become
significantly above 0 at the 70% threshold (fig7c-backbone-pairs-above-random, p = 0.038); and
slice backbone units no longer exceed primary-culture ones at 90% (fig7b-normalized-b2b, p =
0.056). Only the supp10b change bears on an outcome, as recorded under supp10b; fig7b fails in murine
organoids under both readings.

## Murine organoids without MO3

MO3 has four bursts under the stated detection and 160 of its 234 units are backbone units
(table at the top). Every judged contrast that involves murine organoids, and the model-type
effect of supp10b, is fitted again on reading (a) without MO3, leaving MO1, MO7, MO8 and MO9.


In [ ]:
jx = judged_contrasts(R, exclude=("MO3",))
mo = [k for k in ja if "MO" in k[1] or k[0] == "supp10b-rank-order-by-model"]
jm = pd.DataFrame({"with_MO3_estimate": {k: ja[k][0] for k in mo}, "with_MO3_p": {k: ja[k][1] for k in mo},
                   "without_MO3_estimate": {k: jx[k][0] for k in mo}, "without_MO3_p": {k: jx[k][1] for k in mo}})
jm.index.names = ["claim", "contrast"]
print(jm.to_string(float_format=lambda v: f"{v:.3g}"))

Without MO3, murine organoid backbone units have higher normalised burst-to-burst correlations
than non-rigid units (0.075, p = 1e-4, against 0.026, p = 0.11 with MO3), so the murine failure
of fig7b rests on MO3; without it fig7b would meet its partial outcome. Murine organoid lags no
longer differ from slice lags (0.24, p = 0.14), the first contrast of
fig7d-murine-organoids-slices-same-lag; its second contrast, slices against primary cultures, does
not involve murine organoids. The other murine results stay as they were: backbone pairs are not
above 0 (0.020, p = 0.23), backbone three-PC variance is not above non-rigid or all units (p =
0.095 and 0.23), primary backbone units do not fire faster than murine organoid ones (p = 0.69),
and human and murine organoid lags do not differ (p = 0.13). The z-scored supp10b effects stay
significant (p = 0.014 and 0.027). The outcomes are kept as tested on the five murine organoids
with bursts: the paper's code lists backbone units for all nine, so dropping MO3 is not the
paper's choice either. The murine organoid results may come from the burst detection parameters
the paper does not give for these organoids.

## supp27-d2-near-critical

> A subset of every preparation type shows near-critical temporal dynamics (d2 between 0 and 0.1),
> and shuffling spike times raises d2 well above that.

d2 as the Supplementary Methods ("d2 calculation") and the code of Sooter et al. that the paper's
`r_download.sh` fetches (figshare article 25927081, version 1), ported to `sequences.py`:
population counts in 30 ms bins, z-scored, an order-20 AR model by Yule-Walker, and the distance
of its coefficients from the beta = 2 hyperplane. The control moves each unit's spikes to random
frames independently (one shuffle per recording). Recordings at the code's lengths. The AR fit
check is read two ways (see `ar_fit_check`): every log-binned density and spectrum ratio within
one order of magnitude (i), or their median (ii).

In [ ]:
d2 = pd.DataFrame([dict(rec=k, type=R[k]["mtype"], **R[k]["d2"]) for k in RECS])
d2["pass_i"] = (d2[["size_max", "duration_max", "spectrum_max"]] <= 1).all(axis=1)
d2["pass_ii"] = (d2[["size_median", "duration_median", "spectrum_median"]] <= 1).all(axis=1)
print(d2.round(3).to_string(index=False))
w = stats.ttest_ind(d2.d2_shuffled, d2.d2, equal_var=False)
print(f"\nIntact d2: mean {d2.d2.mean():.3f}, range {d2.d2.min():.3f} to {d2.d2.max():.3f}; shuffled: mean {d2.d2_shuffled.mean():.3f}, "
      f"range {d2.d2_shuffled.min():.3f} to {d2.d2_shuffled.max():.3f}")
print(f"Welch t-test, shuffled against intact over the {len(d2)} recordings: t {w.statistic:.1f}, p {w.pvalue:.2g}")
print("Intact recordings with d2 at most 0.1:", int((d2.d2 <= 0.1).sum()), "of", len(d2))
cnt = d2.groupby("type")[["pass_i", "pass_ii"]].sum().astype(int)
cnt["paper"] = pd.Series({"HO": 3, "MO": 7, "MS": 4, "PR": 7})
print("\nRecordings passing the AR fit check, by type:")
print(cnt.to_string())
print("Kept intact recordings with d2 at most 0.1: reading (i)", bool((d2.d2[d2.pass_i] <= 0.1).all()),
      "; reading (ii)", bool((d2.d2[d2.pass_ii] <= 0.1).all()))

**Result: partially reproduced, on the assumption of how the AR fit check is read.** Shuffling
raises d2 in every recording: intact d2 has mean 0.044 (0.014 to 0.099), shuffled 0.221 (0.199 to
0.251), Welch t = 38.3, p = 5e-38. All 31 intact recordings have d2 at most 0.1, so the kept
recordings do under either reading of the fit check. The numbers kept differ from the paper's (3,
7, 4 and 7 for HO, MO, MS and PR): 0, 5, 0 and 1 when every density and spectrum ratio must be
within one order of magnitude, 7, 9, 3 and 8 when their median must be. The paper does not say
how the comparison was made, so the counts are open (a question for the authors).

## The two sets of human organoids (ed2d, ed2e, ed2f)

In [ ]:
for name, recs in [("Sharf (HO1-HO4)", HO4), ("Alam El Din (HO5-HO8)", ["HO5", "HO6", "HO7", "HO8"])]:
    u = unit_table(recs)
    d = u[u.n_spk >= sq.MIN_SPIKES].assign(lr=lambda x: np.log10(x.rate))
    f = sq.lmm(d, "lr ~ bb", "rec")
    print(f"{name}: {len(d)} units ({int(d.bb.sum())} backbone); backbone minus non-rigid log10 rate {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: not reproduced.** Backbone units fire faster in the Sharf recordings (0.72 log10
units, p = 2e-50), and also in the Alam El Din recordings (0.12, p = 0.0019), where the paper
finds no difference (P = 0.40).

In [ ]:
for name, recs in [("Sharf (HO1-HO4)", HO4), ("Alam El Din (HO5-HO8)", ["HO5", "HO6", "HO7", "HO8"])]:
    u = unit_table(recs)
    d = u[np.isfinite(u.b2b30) & np.isfinite(u.b2b_shuf)].assign(nb=lambda x: sq.normalize(x.b2b30, x.b2b_shuf))
    f = sq.lmm(d, "nb ~ bb", "rec")
    print(f"{name}: {len(d)} units; normalised b2b backbone {d[d.bb == 1].nb.mean():.3f}, non-rigid {d[d.bb == 0].nb.mean():.3f}; "
          f"effect {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: reproduced.** Normalised burst-to-burst correlations are higher for backbone units in
both sets (Sharf 0.040, p = 0.005; Alam El Din 0.119, p = 1e-8).

In [ ]:
for name, recs in [("Sharf (HO1-HO4)", HO4), ("Alam El Din (HO5-HO8)", ["HO5", "HO6", "HO7", "HO8"])]:
    out = {}
    for t in ["bb", "mix", "nr"]:
        d = pa[(pa.slice_key == "bb") & pa.rec.isin(recs) & (pa.type == t)]
        if d.rec.nunique() < 2:
            out[t] = dict(pairs=len(d))
            continue
        f = sq.lmm(d, "norm ~ 1", "rec")
        est, p2 = f.params["Intercept"], f.pvalues["Intercept"]
        out[t] = dict(pairs=len(d), recordings=d.rec.nunique(), mean=round(est, 4),
                      p_one_sided=float(f"{(p2 / 2 if est > 0 else 1 - p2 / 2):.2g}"))
    print(name)
    print(pd.DataFrame(out).T.to_string())

**Result: partially reproduced.** Backbone pairs are above 0 in both sets (Sharf 0.148, p =
0.010; Alam El Din 0.185, p = 0.041), and non-rigid pairs are not; mixed pairs are also above 0
in the Sharf recordings (0.087, p = 0.048), where the paper finds P = 0.145.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| methods-burst-count | reproduced (mean only; recordings range from 0 to 156 bursts) |
| fig1c-example-burst-count | reproduced (HO1, 46 bursts) |
| ed1c-lognormal-rates | reproduced (reading a, all units) |
| fig1d-burst-duration | reproduced |
| fig1e-peaks-near-burst-peak | partially reproduced (15 of 16 readings) |
| fig2-backbone-fraction | reproduced (units with at least 30 spikes) |
| fig2e-b2b-correlation | reproduced |
| fig2g-variance-latency | reproduced |
| ed3f-burst-cluster-variability | reproduced |
| ed4b-gabazine-more-bursts | reproduced |
| ed4b-nbqx-cpp-abolish-bursts | reproduced |
| ed4c-gabazine-participation | reproduced |
| ed4d-gabazine-rank-order | reproduced (assumed: z-score normalisation); proposed: blocked, methods_underspecified |
| supp2a-backbone-fraction-declines | partially reproduced (human only) |
| supp2c-b2b-increases-with-age | partially reproduced (human only) |
| supp8-backbone-stable-4h | reproduced (assumed: units linked by position) |
| ed5-shuffle-reduces-b2b | reproduced |
| fig3e-pairwise-correlation | reproduced |
| fig3f-backbone-pairs-in-tail | partially reproduced |
| fig4c-similarity-variance | reproduced |
| fig4d-burst-similarity | reproduced |
| fig4e-similarity-rises-at-backbone-onset | partially reproduced (HO7 has no backbone) |
| supp13-top-correlated-units-similarity | reproduced |
| fig4f-pca-variance | partially reproduced |
| fig6f-slice-b2b-correlation | reproduced |
| fig6g-slice-pairwise-correlation | reproduced |
| fig7a-backbone-higher-rate | reproduced |
| fig7a-primary-higher-rate | partially reproduced |
| fig7b-normalized-b2b | not reproduced (murine organoids) |
| fig7b-primary-no-b2b-difference | reproduced |
| fig7c-normalized-pairwise | reproduced |
| fig7c-slices-primaries-no-pair-difference | reproduced |
| fig7c-backbone-pairs-above-random | not reproduced |
| fig7d-lag-times | partially reproduced |
| fig7d-murine-organoids-slices-same-lag | not reproduced (assumed: zero lags dropped, as the code) |
| fig7e-pca-backbone-variance | not reproduced |
| supp24c-nonrigid-more-variable | reproduced |
| supp10b-rank-order-by-model | reproduced (assumed: z-score normalisation, code's lengths); proposed: blocked, methods_underspecified |
| supp27-d2-near-critical | partially reproduced (assumed: reading of the AR fit check) |
| ed2d-alam-el-din-rates | not reproduced |
| ed2e-alam-el-din-b2b | reproduced |
| ed2f-backbone-pairs-above-zero | partially reproduced |
| HMM claims (supp18, fig5c, fig5d, fig5f, fig7f, supp24d, supp26) | in `hmm_van_der_molen_2025.ipynb` |
| supp5-murine-organoid-forebrain-identity | blocked (data elsewhere; markers not listed) |

**How the analysis differs from the paper's.** 10 shuffles per recording (3 for pairwise
correlations and shuffled similarity; 5 and 2 under the second recording length) where the paper
uses 100; burst-only rates that do not hold the inverse ISI across the gap between bursts (the
code's rates are shown for fig4f); overlapping bursts merged at 700 ms where the paper raises the
peak distance for some recordings; mixed models fitted here by REML with Satterthwaite t tests in
place of the paper's R; MATLAB smoothing replaced by its scipy counterparts (Gaussian with
standard deviation one fifth of the window); the shuffles cover the analysed segment (the code
shuffles the first 180 s); d2 from a Python port of the code of Sooter et al., with one shuffle per
recording and an AR fit check whose rule the paper does not give.

**What was found about the dataset.** The 31 main recordings (8 human organoids, 9 murine organoids,
6 slices, 8 primary cultures) are in the dandiset as sorted files with 1 ms spike matrices, and
HO1 has the 46 bursts of Fig. 1C. The HO2 and HO3 longitudinal recordings are deposited two or three times. The baseline
sorted file of MO12 and all murine organoid longitudinal recordings are missing. Units carry
positions but no identifier linking them across recordings of one organoid; the 0 h and 4 h
recordings of HO2 and HO3, and the baseline and gabazine recordings of MO10, MO11 and MO14,
have identical ordered positions. Pr1 to Pr4 are labelled rat, the paper
calls them murine. With the stated burst detection, four murine organoids have no burst (their
highest peaks reach 3.0 to 3.7 times the RMS), so the paper's murine organoid parameters are not
given.

**Partial outcomes.** One rule is applied to every claim. Where the criterion defines a partial
outcome, it decides. Where it does not, a claim is partially reproduced when its effect is present
but falls short of the criterion in one reading, organoid or contrast (fig1e, fig3f, supp26), and
not reproduced when its central comparison fails under the criterion's tests or the data
contradict a part of it, such as a significant effect where the paper reports none (fig5f,
fig7c-backbone-pairs-above-random, fig7d-murine-organoids-slices-same-lag, ed2d).

**Murine organoids.** The murine organoid failures (fig7b, fig7c-backbone-pairs-above-random,
fig7e) and partial results (fig7a-primary-higher-rate, fig7d-lag-times) rest on five of nine
recordings, under burst detection that the paper does not give for murine organoids. Without MO3
(four bursts) the fig7b murine contrast holds and the first fig7d-murine contrast is
non-significant; the others are unchanged. The outcomes are kept as tested on the five.